In [ ]:
# Try multiple NER datasets until one works
from datasets import load_dataset
import torch

ner_raw = None
sample_size = 10000

# Try 1: slvn/pubmed-ner (scriptless)
try:
    print("Trying slvn/pubmed-ner...")
    data = load_dataset("slvn/pubmed-ner", split="train", streaming=True)
    # Convert to our format
    ner_raw = []
    for item in data:
        if len(ner_raw) >= sample_size:
            break
        # Assuming format with tokens and tags
        tokens = item.get('tokens', [])
        tags = item.get('tags', [])
        if tokens and tags:
            text = " ".join(tokens)
            # Find entities
            i = 0
            while i < len(tokens):
                if tags[i] != 'O':  # Non-'O' tag indicates entity
                    ent_type = tags[i].replace('B-', '').replace('I-', '')
                    ent_text = tokens[i]
                    # Collect full entity span
                    j = i + 1
                    while j < len(tokens) and tags[j].startswith('I-'):
                        ent_text += " " + tokens[j]
                        j += 1
                    ner_raw.append({'text': text, 'entities': [{'entity': ent_text, 'type':
ent_type}]})
                    i = j
                else:
                    i += 1
    print(f"Loaded {len(ner_raw)} from slvn/pubmed-ner")
except Exception as e:
    print(f"slvn/pubmed-ner failed: {e}")

# Try 2: conllpp (scriptless version)
if not ner_raw:
    try:
        print("Trying conllpp...")
        data = load_dataset("conllpp", split="train", streaming=True)
        # Similar processing...
        print(f"Loaded {len(ner_raw)} from conllpp")
    except Exception as e:
        print(f"conllpp failed: {e}")

# Fallback: Use synthetic from RE (already working)
if not ner_raw:
    print("Using synthetic NER data from RE dataset...")
    # Your existing synthetic fallback will work

Trying slvn/pubmed-ner...
slvn/pubmed-ner failed: Dataset 'slvn/pubmed-ner' doesn't exist on the Hub or cannot be accessed.
Trying conllpp...


README.md: 0.00B [00:00, ?B/s]

conllpp.py: 0.00B [00:00, ?B/s]

conllpp failed: Dataset scripts are no longer supported, but found conllpp.py
Using synthetic NER data from RE dataset...


In [ ]:

# Then update the train() function call:

# Cell 2: Load saved CoNLL data and train
ner_raw = torch.load('conll_ner_data.pt')
args.sample_size = len(ner_raw)
print(f"Using {len(ner_raw)} CoNLL NER samples")

# Then run the rest of training normally
# ... rest of your training code ...

In [ ]:
"""
Unified NER + Relation Extraction trainer for pplx-embed-context-v1.

Training Pipeline (GLiNER/GLiDRE style):
1. Frozen pplx-embed backbone (encodes both documents AND labels)
2. Task-prefixed labels: "[NER] person", "[REL] founded_by"
3. Cosine similarity + cross-entropy loss
4. Direct label matrix lookup during training (no ANN)
5. RE architecture: entity span pooler (2-layer MLP) → concat → relation FFN

Architecture Changes from Previous Version:
- Removed: Learnable label embeddings (replaced with frozen pplx-embed encoded labels)
- Removed: MomentumQueue (replaced with direct label matrix lookup)
- Removed: InfoNCE/ArcFace loss (replaced with cosine+CE)
- Added: Task prefixes for all labels
- Refactored: RE encoding to use entity span pooler + FFN (no CLS token)
"""

import argparse
import json
import logging
import math
import numpy as np
import os
import re
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset, Subset
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from transformers import logging as transformers_logging
from datasets import load_dataset

# Auth handling
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        from huggingface_hub import login
        login(token=_tok)
except Exception:
    pass

transformers_logging.set_verbosity_error()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# === COLAB-SAFE MULTIPROCESSING FIX ===
# Use 'spawn' instead of 'fork' to avoid CUDA reinit errors in Colab
# Must be set BEFORE any CUDA operations or imports that use CUDA
try:
    import multiprocessing
    multiprocessing.set_start_method('spawn', force=True)
except RuntimeError:
    # Method already set - ignore
    pass

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(message)s",
                    datefmt="%H:%M:%S", handlers=[logging.StreamHandler(sys.stdout)])
logger = logging.getLogger(__name__)

def char_to_token_span(offsets, start_char, end_char):
    start_tok = end_tok = 1
    for i, (s, e) in enumerate(offsets):
        if e == 0: continue
        if s <= start_char < e: start_tok = i
        if s < end_char <= e: end_tok = i
    return start_tok, max(start_tok, end_tok)


class LoRALinear(nn.Module):
    """LoRA adapter that wraps a linear layer for efficient fine-tuning.

    LoRA (Low-Rank Adaptation) adds trainable rank decomposition matrices to
    the original weight matrix. This allows task-specific adaptation without
    modifying the base embeddings, preserving retrieval quality.

    The adapter can be disabled (set enabled=False) to get pure backbone output.
    """
    def __init__(self, base_layer, rank=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base_layer = base_layer
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.enabled = True  # Can be disabled for pure retrieval

        hidden = base_layer.in_features
        out_features = base_layer.out_features

        # LoRA matrices: A (rank x hidden) and B (out_features x rank)
        # Initialize A randomly and B to zero for smooth training start
        self.lora_A = nn.Parameter(torch.randn(rank, hidden) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

        # Optional dropout for regularization
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        if not self.enabled:
            return self.base_layer(x)

        # Base output
        base_out = self.base_layer(x)

        # LoRA output: x @ A^T @ B^T * scaling
        # Shape: (B, H) @ (rank, H)^T @ (out, rank)^T = (B, out)
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        lora_out = self.dropout(lora_out)

        return base_out + lora_out

    def disable(self):
        """Disable LoRA adapter for pure retrieval mode."""
        self.enabled = False

    def enable(self):
        """Enable LoRA adapter for task-specific mode."""
        self.enabled = True

# Task prefixes for label encoding
NER_PREFIX = "[NER] "
REL_PREFIX = "[REL] "
LINK_PREFIX = "[LINK] "

# Default heldout labels for zero-shot testing
# These are common types that test semantic generalization
DEFAULT_HELDOUT_NER = {"LOC"}  # Hold out locations - test if model learns "place" semantics
DEFAULT_HELDOUT_RE = {
    "P150",           # Wikidata property for "contains administrative territorial entity"
    "located_in",     # Spatial relation
    "founded_by",     # Organizational relation
}  # Diverse relation types to test zero-shot capability

# Default heldout EL ratio - since EL has thousands of entity labels,
# we hold out a larger fraction to test generalization to unseen KB entities
DEFAULT_HELDOUT_EL_RATIO = 0.15  # 15% of KB entities held out


class GenerativeDecoder(nn.Module):
    """
    GLiNER-inspired decoder for generating new labels.

    Takes span embedding as query, produces dual outputs:
    1. Label embedding (D=1024) for vector DB write-back
    2. Label string via text generation head

    Enables "cold path" - generating new labels when no existing label matches
    (theta_null <= score < theta). This enables open-vocabulary expansion.
    """

    def __init__(self, hidden_dim=1024, label_dim=1024, vocab_size=32000,
                 max_decode_len=16, num_heads=8, num_layers=2):
        super().__init__()

        # Task embeddings for conditioning (NER=0, REL=1, LINK=2)
        self.task_embeddings = nn.Embedding(3, hidden_dim)

        # Learned memory for cross-attention (label generation prior)
        # This provides a learned prior for label generation
        self.memory_keys = nn.Parameter(torch.randn(128, hidden_dim) * 0.02)
        self.memory_values = nn.Parameter(torch.randn(128, hidden_dim) * 0.02)

        # Query projection
        self.query_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.GELU(),
            nn.LayerNorm(hidden_dim * 2),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )

        # Cross-attention layers (TransformerDecoder)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=0.1,
            activation='gelu',
            batch_first=True,
        )
        self.decoder_layers = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers,
        )

        # Dual output heads
        # 1. Embedding head: produces label embedding in D=1024 space for vector DB
        self.embedding_head = nn.Sequential(
            nn.Linear(hidden_dim, label_dim),
            nn.LayerNorm(label_dim),
        )

        # 2. Text head: produces logits for label string generation
        self.text_head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, span_emb, task_id):
        """
        Args:
            span_emb: (B, D) span embedding from main model
            task_id: (B,) task index (0=NER, 1=REL, 2=LINK)

        Returns:
            label_emb: (B, D) normalized label embedding for vector DB
            text_logits: (B, vocab) token logits for label string
        """
        B = span_emb.size(0)
        device = span_emb.device

        # Add task conditioning
        task_emb = self.task_embeddings(task_id)
        query = self.query_proj(span_emb) + task_emb
        query = query.unsqueeze(1)  # (B, 1, D)

        # Memory as sequence (learned prior for label generation)
        memory = torch.stack([self.memory_keys, self.memory_values], dim=0).mean(dim=0)
        memory = memory.unsqueeze(0).expand(B, -1, -1)  # (B, 128, D)

        # Decode through cross-attention layers
        decoded = self.decoder_layers(query, memory)  # (B, 1, D)
        decoded = decoded.squeeze(1)  # (B, D)

        # Generate dual outputs
        label_emb = F.normalize(self.embedding_head(decoded), dim=-1, eps=1e-5)
        text_logits = self.text_head(decoded)  # (B, vocab)

        return label_emb, text_logits


def select_heldout_labels(label2idx, task_type="NER",
                         heldout_ratio=0.2, heldout_labels=None, seed=42):
    """Select labels to hold out for zero-shot testing.

    Args:
        label2idx: Dict mapping label names to indices
        task_type: "NER", "RE", or "EL" - selects appropriate defaults
        heldout_ratio: Fraction of labels to hold out (if heldout_labels not specified)
        heldout_labels: Optional explicit list of labels to hold out (overrides defaults)
        seed: Random seed for reproducibility

    Returns:
        seen_labels: Set of labels used during training
        heldout_labels: Set of labels held out for zero-shot testing
    """
    import random
    random.seed(seed)

    all_labels = set(label2idx.keys())

    # If explicit heldout labels provided, use them
    if heldout_labels is not None:
        if isinstance(heldout_labels, str):
            heldout_labels = [heldout_labels]
        # For EL, add the [LINK] prefix if not already present
        if task_type == "EL":
            heldout_labels = [
                l if l.startswith(LINK_PREFIX) else f"{LINK_PREFIX}{l}"
                for l in heldout_labels
            ]
        heldout_set = set(heldout_labels)

        # Filter to only labels that exist in the dataset
        heldout_set = heldout_set.intersection(all_labels)

        if heldout_set:
            seen_labels = all_labels - heldout_set
            print(f"[ZERO-SHOT] Using explicit heldout {task_type} labels: {heldout_set}")
            return seen_labels, heldout_set

    # Try defaults first (NER and RE have explicit defaults; EL uses random sampling)
    if task_type == "NER":
        defaults = {f"{NER_PREFIX}{l}" for l in DEFAULT_HELDOUT_NER}
    elif task_type == "RE":
        defaults = {f"{REL_PREFIX}{l}" for l in DEFAULT_HELDOUT_RE}
    else:  # EL - no explicit defaults, use ratio-based sampling
        defaults = set()

    # Filter defaults to labels that exist
    default_heldout = defaults.intersection(all_labels)

    if default_heldout:
        seen_labels = all_labels - default_heldout
        if task_type == "NER":
            print(f"[ZERO-SHOT] Default heldout {task_type} labels: {DEFAULT_HELDOUT_NER}")
        else:
            print(f"[ZERO-SHOT] Default heldout {task_type} labels: {DEFAULT_HELDOUT_RE}")
        print(f"[ZERO-SHOT] Actual heldout {task_type} labels (with prefix): {default_heldout}")
        return seen_labels, default_heldout

    # Fall back to random sampling (default path for EL)
    num_heldout = max(1, int(len(all_labels) * heldout_ratio))
    heldout_set = set(random.sample(list(all_labels), num_heldout))
    seen_labels = all_labels - heldout_set

    print(f"[ZERO-SHOT] Randomly sampled {num_heldout}/{len(all_labels)} heldout {task_type} labels")
    return seen_labels, heldout_set


def remap_label_indices(label_indices, seen_indices_list):
    """Remap label indices from full label space to seen-only space.

    Args:
        label_indices: Tensor of label indices in full label space
        seen_indices_list: List of seen label indices in full label space

    Returns:
        Remapped tensor of indices in seen-only label space
    """
    # Create mapping from full index to seen-only index
    index_map = {full_idx: seen_idx for seen_idx, full_idx in enumerate(seen_indices_list)}

    # Vectorized remapping
    remapped = torch.tensor([
        index_map.get(idx.item(), 0)  # Default to 0 if label not in seen (shouldn't happen)
        for idx in label_indices
    ], device=label_indices.device)

    return remapped

class UnifiedKGModel(nn.Module):
    def __init__(self, model_name="perplexity-ai/pplx-embed-context-v1-0.6b", label_dim=512,
                 ner_label2idx=None, re_label2idx=None, el_label2idx=None, tokenizer=None, device=None,
                 use_lora=True, lora_rank=8, lora_alpha=16):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        self.hidden = self.backbone.config.hidden_size if hasattr(self.backbone.config, "hidden_size") else 768
        self.label_dim = label_dim
        self.use_lora = use_lora

        # Apply LoRA to backbone attention layers if enabled
        if use_lora:
            self._apply_lora_to_backbone(lora_rank, lora_alpha)

        # Task prefixes for label encoding
        self.ner_prefix = NER_PREFIX
        self.rel_prefix = REL_PREFIX
        self.link_prefix = LINK_PREFIX

        # Label to index mappings
        self.ner_label2idx = ner_label2idx or {}
        self.re_label2idx = re_label2idx or {}
        self.el_label2idx = el_label2idx or {}

        self.num_ner_labels = len(self.ner_label2idx) if self.ner_label2idx else 1
        self.num_re_labels = len(self.re_label2idx) if self.re_label2idx else 1
        self.num_el_labels = len(self.el_label2idx) if self.el_label2idx else 1

        # Initialize projection layers with Xavier for better training dynamics
        # NER projection: 3-layer deeper network with dropout for regularization
        # CRITICAL: Separate projection for SPAN embeddings vs LABEL embeddings
        # This prevents tension between the two objectives
        self.ner_proj = nn.Sequential(
            nn.Linear(self.hidden, self.hidden * 2),
            nn.GELU(),
            nn.LayerNorm(self.hidden * 2),
            nn.Dropout(0.1),
            nn.Linear(self.hidden * 2, label_dim),
            nn.LayerNorm(label_dim),
        )
        # Separate lightweight projection for NER label embeddings
        self.ner_label_proj = nn.Linear(self.hidden, label_dim)
        # Xavier init (eye_ is broken when hidden > label_dim -- truncated identity)
        nn.init.xavier_uniform_(self.ner_label_proj.weight, gain=0.1)
        if self.ner_label_proj.bias is not None:
            nn.init.zeros_(self.ner_label_proj.bias)

        # RE projection: simplified architecture to reduce overfitting
        # Removed complex entity_span_pooler and 3-layer FFN
        # Now: simple concat -> 2-layer projection (matches NER complexity)
        self.relation_proj = nn.Sequential(
            nn.Linear(self.hidden * 2, self.hidden * 2),
            nn.GELU(),
            nn.Dropout(0.15),  # Increased dropout for better regularization
            nn.Linear(self.hidden * 2, label_dim),
        )

        # === TASK-SPECIFIC ADAPTER LAYERS ===
        # Lightweight bottleneck adapters (2-layer, 64 hidden) before projections
        # These allow task-specific transformations without changing backbone
        # NER adapter: adds NER-specific capacity to hidden states
        self.ner_adapter = nn.Sequential(
            nn.Linear(self.hidden, 64),
            nn.GELU(),
            nn.Linear(64, self.hidden)
        )
        # RE adapter: adds RE-specific capacity to hidden states
        self.re_adapter = nn.Sequential(
            nn.Linear(self.hidden, 64),
            nn.GELU(),
            nn.Linear(64, self.hidden)
        )
        # Initialize adapters with near-identity for smooth training
        for adapter in [self.ner_adapter, self.re_adapter]:
            # First layer: small random init
            nn.init.xavier_uniform_(adapter[0].weight, gain=0.1)
            nn.init.zeros_(adapter[0].bias)
            # Second layer: near-identity (output ~ input)
            nn.init.xavier_uniform_(adapter[2].weight, gain=0.1)
            nn.init.zeros_(adapter[2].bias)
        # Separate lightweight projection for RE label embeddings
        self.re_label_proj = nn.Linear(self.hidden * 2, label_dim)
        # Xavier init (eye_ is broken when hidden*2 > label_dim -- truncated identity)
        nn.init.xavier_uniform_(self.re_label_proj.weight, gain=0.1)
        if self.re_label_proj.bias is not None:
            nn.init.zeros_(self.re_label_proj.bias)

        # === ENTITY LINKING COMPONENTS ===
        # EL shares the same span pooler architecture as NER
        # but has separate projection and label embeddings
        self.el_adapter = nn.Sequential(
            nn.Linear(self.hidden, 64),
            nn.GELU(),
            nn.Linear(64, self.hidden)
        )
        # EL projection: similar depth to NER but separate parameters
        self.el_proj = nn.Sequential(
            nn.Linear(self.hidden, self.hidden * 2),
            nn.GELU(),
            nn.LayerNorm(self.hidden * 2),
            nn.Dropout(0.1),
            nn.Linear(self.hidden * 2, label_dim),
            nn.LayerNorm(label_dim),
        )
        # EL label projection
        self.el_label_proj = nn.Linear(self.hidden, label_dim)
        # Xavier init (eye_ is broken when hidden > label_dim -- truncated identity)
        nn.init.xavier_uniform_(self.el_label_proj.weight, gain=0.1)
        if self.el_label_proj.bias is not None:
            nn.init.zeros_(self.el_label_proj.bias)

        # Mention classifier (Stage 1 binary gate)
        # Distinguishes: NIL (expand KB) vs generic/anaphoric (drop)
        self.mention_gate = nn.Sequential(
            nn.Linear(self.hidden, self.hidden // 4),
            nn.ReLU(),
            nn.Linear(self.hidden // 4, 1),
            nn.Sigmoid()
        )

        # === GENERATIVE DECODER (Cold Path - Phase 3 of architecture) ===
        # Enables generation of new labels when no existing label matches
        # Takes span embeddings as input via cross-attention, produces dual outputs:
        # 1. Label embedding (D=1024) for vector DB write-back
        # 2. Human-readable label string
        # Decoder is initialized but not used until train_decoder() is called
        self.decoder = None  # Will be initialized in initialize_decoder()
        self._decoder_initialized = False

        # Apply Xavier initialization to projection layers
        self._init_weights(self.ner_proj)
        self._init_weights(self.relation_proj)  # Simplified RE projection
        # Label projections already initialized with near-identity above

        # Frozen label embeddings (to be initialized after tokenizer/device available)
        self.ner_label_embeddings = None
        self.re_label_embeddings = None
        self.el_label_embeddings = None

        # Store tokenizer/device for deferred label encoding
        self._tokenizer = tokenizer
        self._device = device

        # Temperature annealing: start high for soft gradients, end low for sharp discrimination
        # Using cosine annealing for gentler decay (prevents premature overconfidence)
        # Range increased from 1.5->0.1 to 1.0->0.07 for better stability
        self.temperature_start = 1.0
        self.temperature_end = 0.07
        self.temperature = self.temperature_start

    def _apply_lora_to_backbone(self, rank=8, alpha=16):
        """Apply LoRA adapters to backbone attention QKV projections.

        This adds task-specific capacity without changing base embeddings,
        preserving retrieval quality while enabling NER/RE adaptation.
        """
        lora_count = 0
        for name, module in self.backbone.named_modules():
            # Target Q, K, V projections in attention layers
            if isinstance(module, nn.Linear) and any(x in name for x in ['q_proj', 'k_proj', 'v_proj', 'query', 'key', 'value']):
                # Get parent module
                parent_name = ".".join(name.split(".")[:-1])
                child_name = name.split(".")[-1]

                if parent_name:
                    try:
                        parent = self.backbone.get_submodule(parent_name)
                        wrapped = LoRALinear(module, rank=rank, alpha=alpha, dropout=0.0)
                        setattr(parent, child_name, wrapped)
                        lora_count += 1
                    except Exception as e:
                        # Skip if we can't wrap (e.g., parameter sharing)
                        pass

        print(f"[LORA] Applied LoRA to {lora_count} backbone layers (rank={rank}, alpha={alpha})")

    def set_lora_enabled(self, enabled):
        """Enable or disable LoRA adapters.

        Disable for pure retrieval mode (preserves base embeddings).
        Enable for NER/RE tasks (adds task-specific capacity).
        """
        if not self.use_lora:
            return

        for module in self.backbone.modules():
            if isinstance(module, LoRALinear):
                if enabled:
                    module.enable()
                else:
                    module.disable()

        mode_str = "enabled" if enabled else "disabled"
        print(f"[LORA] Adapters {mode_str} for {'task-specific' if enabled else 'pure retrieval'} mode")

    def _init_weights(self, module):
        """Xavier initialization for projection layers."""
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)

    def encode_frozen_labels(self, label_texts, tokenizer, device):
        """Encode labels through frozen backbone once at init.

        Args:
            label_texts: List of label strings (e.g., ["[NER] person", "[NER] organization"])
            tokenizer: Tokenizer for encoding text
            device: Device to create tensors on

        Returns:
            nn.Parameter with frozen normalized embeddings, requires_grad=False
        """
        if not label_texts:
            return nn.Parameter(torch.empty(0, self.label_dim), requires_grad=False)

        inputs = tokenizer(label_texts, padding=True, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = self.backbone(**inputs)
            # Mean pooling over tokens
            mask = inputs['attention_mask'].unsqueeze(-1).float()
            pooled = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            # Normalize to unit length
            normalized = F.normalize(pooled, dim=-1, eps=1e-5)

        return nn.Parameter(normalized, requires_grad=False)

    def initialize_frozen_labels(self, tokenizer, device):
        """Initialize learnable label embeddings using pplx-embed backbone.

        Must be called after model creation since we need tokenizer/device.
        Labels are encoded ONCE with task prefixes through the backbone,
        then stored as trainable parameters for fine-tuning.

        Note: label2idx already contains prefixed labels from dataset building.
        """
        # The label2idx keys already have prefixes from the dataset
        # So we can use them directly without adding prefix again
        ner_label_texts = list(sorted(self.ner_label2idx.keys()))
        rel_label_texts = list(sorted(self.re_label2idx.keys()))
        el_label_texts = list(sorted(self.el_label2idx.keys()))

        print(f"[INIT] Encoding {len(ner_label_texts)} NER labels with pplx-embed...")
        print(f"[INIT] NER labels: {ner_label_texts[:10]}{'...' if len(ner_label_texts) > 10 else ''}")

        print(f"[INIT] Encoding {len(rel_label_texts)} REL labels with pplx-embed...")
        print(f"[INIT] REL labels: {rel_label_texts[:10]}{'...' if len(rel_label_texts) > 10 else ''}")

        if el_label_texts:
            print(f"[INIT] Encoding {len(el_label_texts)} EL labels with pplx-embed...")
            print(f"[INIT] EL labels: {el_label_texts[:10]}{'...' if len(el_label_texts) > 10 else ''}")

        # Encode through backbone (frozen during encoding)
        # Then project through the projection heads to match span embedding dimensions
        inputs_ner = tokenizer(ner_label_texts, padding=True, return_tensors="pt").to(device)
        inputs_rel = tokenizer(rel_label_texts, padding=True, return_tensors="pt").to(device)

        with torch.no_grad():
            # Backbone encoding
            outputs_ner = self.backbone(**inputs_ner)
            outputs_rel = self.backbone(**inputs_rel)

            # Mean pooling
            mask_ner = inputs_ner['attention_mask'].unsqueeze(-1).float()
            pooled_ner = (outputs_ner.last_hidden_state * mask_ner).sum(1) / mask_ner.sum(1).clamp(min=1e-9)

            mask_rel = inputs_rel['attention_mask'].unsqueeze(-1).float()
            pooled_rel = (outputs_rel.last_hidden_state * mask_rel).sum(1) / mask_rel.sum(1).clamp(min=1e-9)

            # CRITICAL: Use SEPARATE projection heads for label embeddings
            # This prevents tension between span and label embedding optimization
            # Label projections are near-identity initialized to preserve pplx-embed semantics
            projected_ner = self.ner_label_proj(pooled_ner)
            projected_rel = self.re_label_proj(
                torch.cat([pooled_rel, pooled_rel], dim=-1)  # 2H for re_label_proj
            )

            # Normalize
            normalized_ner = F.normalize(projected_ner, dim=-1, eps=1e-5)
            normalized_rel = F.normalize(projected_rel, dim=-1, eps=1e-5)

            # EL encoding (if labels available)
            if el_label_texts:
                inputs_el = tokenizer(el_label_texts, padding=True, return_tensors="pt").to(device)
                outputs_el = self.backbone(**inputs_el)
                mask_el = inputs_el['attention_mask'].unsqueeze(-1).float()
                pooled_el = (outputs_el.last_hidden_state * mask_el).sum(1) / mask_el.sum(1).clamp(min=1e-9)
                projected_el = self.el_label_proj(pooled_el)
                normalized_el = F.normalize(projected_el, dim=-1, eps=1e-5)
            else:
                # Empty EL embeddings placeholder
                normalized_el = torch.empty(0, self.label_dim, device=device)

        # Remove the placeholder None attributes before creating parameters
        if hasattr(self, 'ner_label_embeddings'):
            delattr(self, 'ner_label_embeddings')
        if hasattr(self, 're_label_embeddings'):
            delattr(self, 're_label_embeddings')
        if hasattr(self, 'el_label_embeddings'):
            delattr(self, 'el_label_embeddings')

        # Register as FROZEN BASE PARAMETERS for zero-shot generalization
        # Keeping label embeddings frozen preserves pplx-embed semantics and enables
        # open-vocabulary retrieval of new labels via semantic similarity
        self.ner_label_embeddings_base = nn.Parameter(normalized_ner.clone(), requires_grad=False)
        self.re_label_embeddings_base = nn.Parameter(normalized_rel.clone(), requires_grad=False)
        self.el_label_embeddings_base = nn.Parameter(normalized_el.clone(), requires_grad=False)

        # Add small LEARNABLE ADAPTERS to allow task-specific adaptation
        # while preserving the semantic structure of the frozen embeddings
        # Small initialization (0.01) ensures minimal impact on semantics initially
        self.ner_label_adapter = nn.Parameter(
            torch.zeros_like(normalized_ner) * 0.01,
            requires_grad=True
        )
        self.re_label_adapter = nn.Parameter(
            torch.zeros_like(normalized_rel) * 0.01,
            requires_grad=True
        )
        if normalized_el.shape[0] > 0:
            self.el_label_adapter = nn.Parameter(
                torch.zeros_like(normalized_el) * 0.01,
                requires_grad=True
            )
        else:
            self.el_label_adapter = None

        # Combined label embeddings (base + adapter, normalized)
        # This is computed dynamically in get_label_embeddings()
        self.ner_label_embeddings = None
        self.re_label_embeddings = None
        self.el_label_embeddings = None

        # Verify frozen base and trainable adapters
        print(f"[INIT] NER base embeddings frozen (requires_grad={self.ner_label_embeddings_base.requires_grad})")
        print(f"[INIT] REL base embeddings frozen (requires_grad={self.re_label_embeddings_base.requires_grad})")
        print(f"[INIT] EL base embeddings frozen (requires_grad={self.el_label_embeddings_base.requires_grad})")
        print(f"[INIT] NER adapter trainable (requires_grad={self.ner_label_adapter.requires_grad})")
        print(f"[INIT] REL adapter trainable (requires_grad={self.re_label_adapter.requires_grad})")
        if self.el_label_adapter is not None:
            print(f"[INIT] EL adapter trainable (requires_grad={self.el_label_adapter.requires_grad})")
        print(f"[INIT] Adapters enable task-specific adaptation while preserving pplx-embed semantics")

        return self._get_combined_label_embeddings()

    def _get_combined_label_embeddings(self):
        """Get combined label embeddings (frozen base + learnable adapter).

        The adapter allows task-specific adaptation while the frozen base preserves
        pplx-embed semantics for zero-shot transfer.

        Returns:
            ner_embeddings, re_embeddings, el_embeddings
        """
        ner_emb = F.normalize(
            self.ner_label_embeddings_base + self.ner_label_adapter,
            dim=-1, eps=1e-5
        )
        re_emb = F.normalize(
            self.re_label_embeddings_base + self.re_label_adapter,
            dim=-1, eps=1e-5
        )
        if self.el_label_adapter is not None:
            el_emb = F.normalize(
                self.el_label_embeddings_base + self.el_label_adapter,
                dim=-1, eps=1e-5
            )
        else:
            el_emb = self.el_label_embeddings_base

        # Cache for backward compatibility
        self.ner_label_embeddings = ner_emb
        self.re_label_embeddings = re_emb
        self.el_label_embeddings = el_emb

        return ner_emb, re_emb, el_emb

    def initialize_decoder(self, tokenizer, device):
        """Initialize the generative decoder for cold path label generation.

        The decoder is initialized separately from the main model since it's trained
        in a separate phase after the main NER/RE/EL heads are frozen.

        Args:
            tokenizer: Tokenizer for text generation head
            device: Device to create tensors on
        """
        if self._decoder_initialized:
            return self.decoder

        vocab_size = tokenizer.vocab_size if hasattr(tokenizer, 'vocab_size') else 32000

        self.decoder = GenerativeDecoder(
            hidden_dim=self.label_dim,
            label_dim=self.label_dim,
            vocab_size=vocab_size,
            max_decode_len=16,
            num_heads=8,
            num_layers=2,
        ).to(device)

        # Keep decoder frozen until train_decoder is called
        for param in self.decoder.parameters():
            param.requires_grad = False

        self._decoder_initialized = True
        print(f"[DECODER] Initialized generative decoder (frozen, will train in separate phase)")
        print(f"[DECODER] Vocab size: {vocab_size}, Hidden dim: {self.label_dim}")

        return self.decoder

    def _span_pool(self, hidden, starts, ends):
        """Vectorized mean pool over token spans - 10-20x faster than loop."""
        B, T, H = hidden.shape
        device = hidden.device

        # Ensure starts/ends are on same device as hidden
        starts = starts.to(device)
        ends = ends.to(device)

        # Clamp to valid range [0, T-1]
        starts = torch.clamp(starts, 0, T-1)
        ends = torch.clamp(ends, 0, T-1)
        # Ensure ends >= starts
        ends = torch.maximum(ends, starts)

        # Create span mask tensor (B, T) - 1 for tokens in span, 0 otherwise
        token_indices = torch.arange(T, device=device).unsqueeze(0).expand(B, -1)  # (B, T)
        span_mask = (token_indices >= starts.unsqueeze(1)) & (token_indices <= ends.unsqueeze(1))  # (B, T)
        span_lengths = span_mask.sum(dim=1, keepdim=True).clamp(min=1).float()  # (B, 1)

        # Apply mask and sum: (B, T, H) * (B, T, 1) -> (B, H)
        masked_hidden = hidden * span_mask.unsqueeze(-1)  # (B, T, H)
        pooled = masked_hidden.sum(dim=1) / span_lengths  # (B, H)

        # Handle NaN/Inf (empty spans)
        valid = ~(torch.isnan(pooled).any(dim=1) | torch.isinf(pooled).any(dim=1))
        if not valid.all():
            pooled[~valid] = torch.randn(H, device=device) * 0.01

        return pooled

    def encode_spans_ner(self, hidden, starts, ends):
        """Encode NER spans through task adapter + projection layer."""
        pooled = self._span_pool(hidden, starts, ends)
        # Apply NER-specific adapter for task transformation
        adapted = self.ner_adapter(pooled)
        projected = self.ner_proj(adapted)
        if torch.isnan(projected).any() or torch.isinf(projected).any():
            projected = torch.randn_like(projected) * 0.01
        return F.normalize(projected + 1e-8, dim=-1, eps=1e-5)

    def encode_pairs_re(self, hidden, s_start, s_end, o_start, o_end):
        """Encode relation pairs through simplified architecture (reduced overfitting).

        Simplified architecture:
        1. Pool subject and object spans
        2. Pass each through RE adapter for task transformation
        3. Concatenate entity representations (2H only)
        4. Pass through simple 2-layer relation projection (matches NER complexity)

        This simplified architecture reduces overfitting that was causing RE to peak
        at epoch 2 and then degrade.
        """
        # Step 1: Mean pool spans
        subj_pool = self._span_pool(hidden, s_start, s_end)
        obj_pool = self._span_pool(hidden, o_start, o_end)

        # Step 2: Apply RE-specific adapter for task transformation
        subj_adapted = self.re_adapter(subj_pool)
        obj_adapted = self.re_adapter(obj_pool)

        # Step 3: Concatenate entity representations (2H)
        pair_emb = torch.cat([subj_adapted, obj_adapted], dim=-1)

        # Step 4: Simple 2-layer projection (matches NER complexity)
        projected = self.relation_proj(pair_emb)

        if torch.isnan(projected).any() or torch.isinf(projected).any():
            projected = torch.randn_like(projected) * 0.01
        return F.normalize(projected + 1e-8, dim=-1, eps=1e-5)

    def encode_mentions_el(self, hidden, starts, ends):
        """Encode entity linking mentions through adapter + projection.

        EL uses same span pooling as NER but separate adapter/projection.
        The mention_gate filters generic/anaphoric mentions before vector DB lookup.
        """
        pooled = self._span_pool(hidden, starts, ends)
        # Apply EL-specific adapter
        adapted = self.el_adapter(pooled)
        projected = self.el_proj(adapted)
        if torch.isnan(projected).any() or torch.isinf(projected).any():
            projected = torch.randn_like(projected) * 0.01
        return F.normalize(projected + 1e-8, dim=-1, eps=1e-5)

    def forward(self, input_ids, attention_mask, return_all=False,
                ner_spans=None, re_pairs=None):
        """Multi-task single-pass forward pass.

        Args:
            input_ids: (B, T) token IDs
            attention_mask: (B, T) attention mask
            return_all: if True, return all outputs; else just retrieval embeddings
            ner_spans: optional list of (start, end) tuples for NER
            re_pairs: optional list of (s_start, s_end, o_start, o_end) tuples for RE

        Returns:
            If return_all=False: retrieval_emb (B, D)
            If return_all=True: dict with retrieval, ner_spans, re_pairs
        """
        # Single backbone pass
        outputs = self.backbone(input_ids, attention_mask)
        hidden = outputs.last_hidden_state

        # 1. Retrieval embeddings (raw mean pool - unchanged for search)
        mask = attention_mask.unsqueeze(-1).float()
        retrieval_emb = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        retrieval_emb = F.normalize(retrieval_emb, dim=-1, eps=1e-5)

        if not return_all:
            return retrieval_emb

        result = {"retrieval": retrieval_emb}

        # 2. NER spans (through adapters + projection)
        if ner_spans is not None:
            ner_embeddings = []
            for s_start, s_end in ner_spans:
                ner_emb = self.encode_spans_ner(hidden,
                                                 torch.tensor([s_start]).to(hidden.device),
                                                 torch.tensor([s_end]).to(hidden.device))
                ner_embeddings.append(ner_emb)
            result["ner_spans"] = torch.cat(ner_embeddings, dim=0) if ner_embeddings else None

        # 3. RE pairs (through adapters + projection)
        if re_pairs is not None:
            re_embeddings = []
            for s_start, s_end, o_start, o_end in re_pairs:
                re_emb = self.encode_pairs_re(hidden,
                                               torch.tensor([s_start]).to(hidden.device),
                                               torch.tensor([s_end]).to(hidden.device),
                                               torch.tensor([o_start]).to(hidden.device),
                                               torch.tensor([o_end]).to(hidden.device))
                re_embeddings.append(re_emb)
            result["re_pairs"] = torch.cat(re_embeddings, dim=0) if re_embeddings else None

        return result


def arcface_loss(span_embeddings, label_embeddings, label_indices,
                 temperature=0.07, margin_ner=0.05, margin_re=0.0, task_type="NER"):
    """ArcFace loss with adaptive margins per task.

    ArcFace adds angular margin penalty to the positive class only, pushing
    the decision boundary away from positive samples. This creates more
    discriminative embeddings compared to standard cosine + CE.

    Args:
        span_embeddings: (B, D) - projected span representations
        label_embeddings: (L, D) - label embeddings
        label_indices: (B,) - true label indices
        temperature: float - scaling temperature for softmax
        margin_ner: float - angular margin for NER task (radians, smaller is better)
        margin_re: float - angular margin for RE task (radians, smaller is better)
        task_type: str - "NER" or "RE" for margin selection

    Returns:
        loss: scalar - ArcFace loss with label smoothing
        max_sim: scalar - mean cosine similarity for true labels (before margin)
    """
    # Cosine similarity: (B, L)
    cosine_sim = F.cosine_similarity(
        span_embeddings.unsqueeze(1),  # (B, 1, D)
        label_embeddings.unsqueeze(0),  # (1, L, D)
        dim=-1
    )  # (B, L)

    # Select margin based on task type
    # RE has more classes → smaller margin for stability
    margin = margin_ner if task_type == "NER" else margin_re

    # Clamp cosine to valid range for arccos (avoid NaN)
    cosine_sim = torch.clamp(cosine_sim, min=-0.99999, max=0.99999)

    # Convert to angular space: theta = arccos(cos_sim)
    theta = torch.acos(cosine_sim)

    # Apply angular margin to positive class ONLY
    # ArcFace: cos(theta + m) = cos(theta)cos(m) - sin(theta)sin(m)
    num_labels = cosine_sim.size(1)
    one_hot = F.one_hot(label_indices, num_classes=num_labels).float()

    # Add margin only to true class
    theta_m = theta + margin * one_hot

    # Convert back to cosine space
    cosine_sim_m = torch.cos(theta_m)

    # Scale by temperature (no separate scale parameter - avoids double scaling)
    logits = cosine_sim_m / temperature

    # Cross-entropy with increased label smoothing for better regularization
    loss = F.cross_entropy(logits, label_indices, label_smoothing=0.1)

    # Return pre-margin similarity for metrics (what we actually care about)
    max_sim = cosine_sim[torch.arange(len(label_indices)), label_indices].mean()

    return loss, max_sim.item()


def cosine_cross_entropy_loss_chunked(span_embeddings, label_embeddings, label_indices, temperature=0.07, chunk_size=1024):
    """Memory-efficient cosine similarity + cross-entropy for large label sets (e.g., EL with 6988 labels).

    Chunks the label embeddings to avoid OOM during cosine similarity computation.

    Args:
        span_embeddings: (B, D) - projected span representations
        label_embeddings: (L, D) - label embeddings
        label_indices: (B,) - true label indices
        temperature: float - scaling temperature
        chunk_size: int - number of labels to process per chunk

    Returns:
        loss: scalar - cross-entropy loss
        max_sim: scalar - mean cosine similarity for true labels
    """
    num_labels = label_embeddings.size(0)
    batch_size = span_embeddings.size(0)

    # Normalize embeddings once
    span_norm = F.normalize(span_embeddings, p=2, dim=-1)  # (B, D)
    label_norm = F.normalize(label_embeddings, p=2, dim=-1)  # (L, D)

    # Compute cosine similarity in chunks
    all_logits = []
    for start_idx in range(0, num_labels, chunk_size):
        end_idx = min(start_idx + chunk_size, num_labels)
        label_chunk = label_norm[start_idx:end_idx]  # (chunk, D)

        # Matrix mult: (B, D) @ (D, chunk) -> (B, chunk)
        chunk_sim = torch.matmul(span_norm, label_chunk.T)  # (B, chunk)
        all_logits.append(chunk_sim)

    # Concatenate: (B, L)
    cosine_sim = torch.cat(all_logits, dim=1)

    # Clamp to valid cosine similarity range [-1, 1] due to floating-point precision
    # Learned embeddings can drift from exact unit norm during training
    cosine_sim = torch.clamp(cosine_sim, min=-1.0, max=1.0)

    # Get similarity for true labels (for metrics) - BEFORE temperature scaling
    true_sim = cosine_sim[torch.arange(batch_size), label_indices].mean()

    # Scale by temperature for loss only
    logits = cosine_sim / temperature
    # Increased label smoothing for better regularization (0.01 -> 0.1)
    ce_loss = F.cross_entropy(logits, label_indices, label_smoothing=0.1)

    return ce_loss, true_sim.item()


def cosine_cross_entropy_loss(span_embeddings, label_embeddings, label_indices, temperature=0.07):
    """Cosine similarity + cross-entropy loss with label smoothing and hard negative mining.

    Direct matrix lookup (all labels, no ANN during training).

    Args:
        span_embeddings: (B, D) - projected span representations
        label_embeddings: (L, D) - label embeddings (now trainable)
        label_indices: (B,) - true label indices
        temperature: float - scaling temperature for softmax

    Returns:
        loss: scalar - cross-entropy loss with label smoothing + hard negative margin
        max_sim: scalar - mean cosine similarity for true labels
    """
    # Cosine similarity: (B, L)
    cosine_sim = F.cosine_similarity(
        span_embeddings.unsqueeze(1),  # (B, 1, D)
        label_embeddings.unsqueeze(0),  # (1, L, D)
        dim=-1
    )  # (B, L)

    # Clamp to valid cosine similarity range [-1, 1] due to floating-point precision
    # Learned embeddings can drift from exact unit norm during training
    cosine_sim = torch.clamp(cosine_sim, min=-1.0, max=1.0)

    # Scale by temperature for loss
    logits = cosine_sim / temperature

    # Cross-entropy with increased label smoothing for better regularization
    # Even with few classes, smoothing helps prevent overconfidence
    ce_loss = F.cross_entropy(logits, label_indices, label_smoothing=0.1)

    # Hard negative mining: find top-K hardest negatives and push them away
    batch_size = len(label_indices)
    num_labels = cosine_sim.size(1)

    # Get similarities for true labels
    true_sim = cosine_sim[torch.arange(batch_size), label_indices]

    # Mask out true labels to get negatives only
    mask = torch.ones_like(cosine_sim, dtype=torch.bool)
    mask[torch.arange(batch_size), label_indices] = False

    # Get top-K hard negatives (highest similarity but wrong label)
    k = min(5, num_labels - 1)  # Use 5 hard negatives, or fewer if not enough labels
    if k > 0:
        # For each sample, get top-K hard negative similarities
        masked_sim = cosine_sim.masked_fill(~mask, -float('inf'))
        hard_neg_top = masked_sim.topk(k=k, dim=1).values  # (B, K)

        # Hard negative margin loss: push hard negatives away from true label
        # max(0, hard_neg_sim - true_sim + margin) encourages hard_neg_sim < true_sim - margin
        # CRITICAL: Re-enabled hard negative mining with lower weight to reduce noise
        # This is essential for RE with 94 classes - needs fine-grained discrimination
        margin = 0.1
        margin_loss = F.relu(hard_neg_top - true_sim.unsqueeze(1) + margin).mean()

        # Combined loss: CE + weighted hard negative mining
        # Lower weight (0.03) on margin loss reduces noise while still providing discrimination signal
        loss = ce_loss + 0.03 * margin_loss

    else:
        # Fallback when k=0 (not enough labels)
        loss = ce_loss

    # Also return max similarity for metrics (similarity of true label pairs)
    max_sim = cosine_sim[torch.arange(batch_size), label_indices].mean()

    return loss, max_sim.item()


def focal_cosine_loss(span_embeddings, label_embeddings, label_indices,
                      temperature=0.07, gamma=2.0, alpha=0.25):
    """Focal loss with cosine similarity for class-imbalanced learning.

    Focal loss down-weights easy examples, focusing on hard/rare ones.
    FL = -alpha * (1 - p_t)^gamma * log(p_t)

    This is particularly useful for EL which has 6988 labels with many rare types.

    Args:
        span_embeddings: (B, D) - projected span representations
        label_embeddings: (L, D) - ALL label embeddings (frozen, seen + heldout)
        label_indices: (B,) - true label indices in full label space
        temperature: float - scaling temperature for softmax
        gamma: float - focusing parameter (higher = more focus on hard examples)
        alpha: float - weight for focal loss

    Returns:
        loss: scalar - focal loss
        max_sim: scalar - mean cosine similarity for true labels
    """
    batch_size = len(label_indices)

    # Cosine similarity to ALL labels: (B, L)
    cosine_sim = F.cosine_similarity(
        span_embeddings.unsqueeze(1),  # (B, 1, D)
        label_embeddings.unsqueeze(0),  # (1, L, D)
        dim=-1
    )  # (B, L)

    # Clamp to valid range
    cosine_sim = torch.clamp(cosine_sim, min=-1.0, max=1.0)

    # Scale by temperature
    logits = cosine_sim / temperature

    # Log probabilities
    log_probs = F.log_softmax(logits, dim=-1)
    probs = torch.exp(log_probs)

    # Get probabilities for true labels
    true_log_probs = log_probs[torch.arange(batch_size), label_indices]
    true_probs = probs[torch.arange(batch_size), label_indices]

    # Focal term: (1 - p)^gamma - down-weights easy examples
    focal_weight = (1 - true_probs) ** gamma

    # Apply focal weighting
    focal_loss = -alpha * focal_weight * true_log_probs

    return focal_loss.mean(), true_probs.mean().item()


def cosine_cross_entropy_loss_zero_shot(span_embeddings, label_embeddings, label_indices,
                                          seen_indices=None, temperature=0.07,
                                          contrastive_weight=0.05, use_focal=True,
                                          focal_gamma=2.0, focal_alpha=0.25,
                                          label_smoothing=0.1):
    """Zero-shot cosine similarity + cross-entropy loss for open-vocabulary learning.

    Key insight: With frozen pplx-embed label embeddings, we don't need special masking.
    The frozen embeddings already preserve semantic relationships. We just train the
    span projection to align with the frozen label space.

    Args:
        span_embeddings: (B, D) - projected span representations
        label_embeddings: (L, D) - ALL label embeddings (frozen, seen + heldout)
        label_indices: (B,) - true label indices in full label space
        seen_indices: list of indices for seen labels (unused, kept for compatibility)
        temperature: float - scaling temperature for softmax
        contrastive_weight: float - weight for contrastive loss component
        use_focal: bool - whether to use focal loss (better for imbalanced classes)
        focal_gamma: float - focusing parameter for focal loss
        focal_alpha: float - weight for focal loss
        label_smoothing: float - label smoothing for CE (increased to 0.1 for better regularization)

    Returns:
        loss: scalar - cross-entropy loss + optional contrastive loss
        max_sim: scalar - mean cosine similarity for true labels
    """
    batch_size = len(label_indices)

    # Cosine similarity to ALL labels: (B, L)
    cosine_sim = F.cosine_similarity(
        span_embeddings.unsqueeze(1),  # (B, 1, D)
        label_embeddings.unsqueeze(0),  # (1, L, D)
        dim=-1
    )  # (B, L)

    # Clamp to valid range
    cosine_sim = torch.clamp(cosine_sim, min=-1.0, max=1.0)

    # Scale by temperature and compute standard cross-entropy
    # With frozen embeddings, all labels (seen + heldout) participate in softmax
    # This preserves semantic relationships and enables zero-shot transfer
    logits = cosine_sim / temperature

    # Use focal loss for better handling of class imbalance (many rare labels)
    if use_focal:
        log_probs = F.log_softmax(logits, dim=-1)
        probs = torch.exp(log_probs)

        # Get probabilities for true labels
        true_log_probs = log_probs[torch.arange(batch_size), label_indices]
        true_probs = probs[torch.arange(batch_size), label_indices]

        # Focal term: (1 - p)^gamma - down-weights easy examples
        focal_weight = (1 - true_probs) ** focal_gamma

        # Focal loss with label smoothing via soft targets
        # Apply smoothing to the true log probs
        num_classes = logits.shape[-1]
        smooth_true = (1 - label_smoothing) * true_log_probs + label_smoothing * torch.log(torch.ones_like(true_probs) / num_classes)
        ce_loss = -(focal_weight * smooth_true).mean()

        true_sim = cosine_sim[torch.arange(batch_size), label_indices]
    else:
        # Standard cross-entropy loss with increased label smoothing
        ce_loss = F.cross_entropy(logits, label_indices, label_smoothing=label_smoothing)

        # Optional: Small contrastive component to maximize similarity to true label
        # This helps the span projection align better with the frozen label space
        true_sim = cosine_sim[torch.arange(batch_size), label_indices]

    # Combined loss (lower contrastive weight since frozen embeddings already help)
    contrastive_loss = -true_sim.mean()
    loss = ce_loss + contrastive_weight * contrastive_loss

    return loss, true_sim.mean().item()


# === GENERATIVE DECODER LOSS FUNCTIONS ===

def decoder_embedding_loss(generated_emb, true_label_emb, negative_labels, temperature=0.07):
    """Contrastive loss for generated embeddings.

    Ensures generated embeddings match true label embeddings and are pushed away
    from negative label embeddings (InfoNCE loss).

    Args:
        generated_emb: (B, D) embeddings from decoder
        true_label_emb: (B, D) true label embeddings (from backbone)
        negative_labels: (N, D) negative label embeddings
        temperature: Softmax temperature

    Returns:
        loss: Scalar contrastive loss
    """
    # Positive similarity
    pos_sim = F.cosine_similarity(generated_emb, true_label_emb, dim=-1)  # (B,)

    # Negative similarities — handle both shared (N, D) and per-sample (B, N, D) negatives
    if negative_labels.dim() == 3:
        # Per-sample negatives: (B, N, D)
        neg_sim = F.cosine_similarity(
            generated_emb.unsqueeze(1),  # (B, 1, D)
            negative_labels,             # (B, N, D)
            dim=-1
        )  # (B, N)
    else:
        # Shared negatives: (N, D)
        neg_sim = F.cosine_similarity(
            generated_emb.unsqueeze(1),  # (B, 1, D)
            negative_labels.unsqueeze(0),  # (1, N, D)
            dim=-1
        )  # (B, N)

    # InfoNCE loss: positive should be higher than all negatives
    logits = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1) / temperature
    labels = torch.zeros(generated_emb.size(0), dtype=torch.long, device=generated_emb.device)

    return F.cross_entropy(logits, labels)


def decoder_text_loss(text_logits, target_ids, attention_mask):
    """Cross-entropy loss for text generation.

    Non-autoregressive: same prediction for each position in target sequence.

    Args:
        text_logits: (B, vocab) model outputs
        target_ids: (B, max_len) target token IDs
        attention_mask: (B, max_len) mask for padding

    Returns:
        loss: Scalar cross-entropy loss
    """
    B, vocab_size = text_logits.shape

    # Expand logits to match target sequence length
    # For non-autoregressive: same prediction for each position
    max_len = target_ids.size(1)
    text_logits = text_logits.unsqueeze(1).expand(B, max_len, vocab_size)

    # Flatten
    text_logits = text_logits.reshape(-1, vocab_size)
    target_ids = target_ids.reshape(-1)
    mask = attention_mask.reshape(-1).bool()

    # Compute loss only on non-padding tokens
    if mask.sum() > 0:
        loss = F.cross_entropy(text_logits[mask], target_ids[mask])
    else:
        # All padding, return zero loss
        loss = torch.tensor(0.0, device=text_logits.device)

    return loss


def combined_decoder_loss(generated_emb, true_label_emb, negative_labels,
                          text_logits, target_ids, attention_mask,
                          lambda_emb=1.0, lambda_text=1.0):
    """Combined decoder loss with both embedding and text components.

    Args:
        lambda_emb: Weight for embedding loss
        lambda_text: Weight for text generation loss

    Returns:
        total_loss, emb_loss, text_loss
    """
    emb_loss = decoder_embedding_loss(generated_emb, true_label_emb, negative_labels)
    text_loss = decoder_text_loss(text_logits, target_ids, attention_mask)

    total_loss = lambda_emb * emb_loss + lambda_text * text_loss

    return total_loss, emb_loss, text_loss


class DecoderDataset(Dataset):
    """
    Dataset for training the generative decoder.

    Pre-computed span embeddings paired with their label strings.
    Uses cached embeddings from the trained NER/RE/EL heads.
    """

    def __init__(self, embeddings_path, tokenizer, max_length=16):
        """
        Args:
            embeddings_path: Path to cached embeddings dict
            tokenizer: Tokenizer for text generation
            max_length: Max sequence length for generated labels
        """
        cached = torch.load(embeddings_path, weights_only=False)
        self.span_embeddings = cached['span_embeddings']
        self.label_texts = cached['label_texts']
        self.task_ids = cached['task_ids']  # 0=NER, 1=REL, 2=LINK
        self.tokenizer = tokenizer
        self.max_length = max_length

        print(f"[DECODER DATASET] Loaded {len(self.span_embeddings)} samples from {embeddings_path}")

    def __len__(self):
        return len(self.span_embeddings)

    def __getitem__(self, idx):
        span_emb = self.span_embeddings[idx]
        label_text = self.label_texts[idx]
        task_id = self.task_ids[idx]

        # Remove task prefix for generation (decoder adds it back based on task_id)
        prefixes = ["[NER] ", "[REL] ", "[LINK] "]
        for prefix in prefixes:
            if label_text.startswith(prefix):
                label_text = label_text[len(prefix):]
                break

        # Encode target text
        encoded = self.tokenizer(
            label_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'span_emb': span_emb,
            'task_id': task_id,
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
        }


def prepare_decoder_training_data(model, dataloaders, datasets, tokenizer, device, output_path, max_samples=5000):
    """Pre-compute span embeddings and labels for decoder training.

    Should be run AFTER main NER/RE/EL training is complete.
    Collects span embeddings from the frozen main model to train the decoder.

    Args:
        model: Trained UnifiedKGModel with frozen main components
        dataloaders: Dict with 'ner', 're', 'el' dataloaders
        datasets: Dict with 'ner', 're', 'el' UnifiedDataset objects (for idx2label)
        tokenizer: Tokenizer for text processing
        device: Device to run on
        output_path: Path to save cached embeddings
        max_samples: Max samples per task to collect
    """
    model.eval()

    all_span_embs = []
    all_labels = []
    all_task_ids = []

    print("[DECODER PREP] Collecting NER span embeddings...")

    # Get idx2label from datasets
    ner_idx2label = datasets.get('ner', {}).idx2label if datasets.get('ner') else {}
    re_idx2label = datasets.get('re', {}).idx2label if datasets.get('re') else {}
    el_idx2label = datasets.get('el', {}).idx2label if datasets.get('el') else {}

    # Collect from NER data
    count = 0
    for batch in dataloaders.get('ner', []):
        if count >= max_samples:
            break
        with torch.no_grad():
            hidden = model.backbone(batch['input_ids'].to(device),
                                   batch['attention_mask'].to(device)).last_hidden_state
            span_embs = model.encode_spans_ner(hidden, batch['s_start'], batch['s_end'])
        all_span_embs.append(span_embs.cpu())
        # Get label texts from idx2label
        for idx in batch['label_indices']:
            label_text = ner_idx2label.get(idx.item(), f"ner_label_{idx.item()}")
            all_labels.append(label_text)
            all_task_ids.append(0)  # NER task
        count += len(batch['label_indices'])

    print(f"[DECODER PREP] Collected {len([x for x in all_task_ids if x == 0])} NER samples")

    # Collect from RE data
    print("[DECODER PREP] Collecting RE span embeddings...")
    count = 0
    for batch in dataloaders.get('re', []):
        if count >= max_samples:
            break
        with torch.no_grad():
            hidden = model.backbone(batch['input_ids'].to(device),
                                   batch['attention_mask'].to(device)).last_hidden_state
            # Use subject entity span as the query
            span_embs = model.encode_spans_ner(hidden, batch['s_start'], batch['s_end'])
        all_span_embs.append(span_embs.cpu())
        for idx in batch['label_indices']:
            label_text = re_idx2label.get(idx.item(), f"re_label_{idx.item()}")
            all_labels.append(label_text)
            all_task_ids.append(1)  # RE task
        count += len(batch['label_indices'])

    print(f"[DECODER PREP] Collected {len([x for x in all_task_ids if x == 1])} RE samples")

    # Collect from EL data
    if 'el' in dataloaders and dataloaders['el'] is not None:
        print("[DECODER PREP] Collecting EL span embeddings...")
        count = 0
        for batch in dataloaders['el']:
            if count >= max_samples:
                break
            with torch.no_grad():
                hidden = model.backbone(batch['input_ids'].to(device),
                                       batch['attention_mask'].to(device)).last_hidden_state
                span_embs = model.encode_mentions_el(hidden, batch['s_start'], batch['s_end'])
            all_span_embs.append(span_embs.cpu())
            for idx in batch['label_indices']:
                label_text = el_idx2label.get(idx.item(), f"el_label_{idx.item()}")
                all_labels.append(label_text)
                all_task_ids.append(2)  # EL task
            count += len(batch['label_indices'])

        print(f"[DECODER PREP] Collected {len([x for x in all_task_ids if x == 2])} EL samples")

    # Concatenate all embeddings
    all_span_embs = torch.cat(all_span_embs, dim=0)
    all_task_ids = torch.tensor(all_task_ids)

    print(f"[DECODER PREP] Total samples: {len(all_span_embs)}")
    print(f"[DECODER PREP] Span embeddings shape: {all_span_embs.shape}")
    print(f"[DECODER PREP] Task IDs: NER={sum(all_task_ids==0)}, RE={sum(all_task_ids==1)}, EL={sum(all_task_ids==2)}")

    # Save cache
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    torch.save({
        'span_embeddings': all_span_embs,
        'label_texts': all_labels,
        'task_ids': all_task_ids,
    }, output_path)

    print(f"[DECODER PREP] Saved decoder training data to {output_path}")

    return output_path


def train_decoder(model, decoder_data_path, tokenizer, device, args):
    """Train the generative decoder in a separate phase.

    Main model (backbone + NER/RE/EL heads) should be frozen.
    Only decoder parameters are updated during this phase.

    Args:
        model: Trained UnifiedKGModel with decoder initialized
        decoder_data_path: Path to pre-computed decoder training data
        tokenizer: Tokenizer for text processing
        device: Device to run on
        args: Arguments with decoder training hyperparameters

    Returns:
        Training metrics summary
    """
    # Load decoder dataset
    decoder_dataset = DecoderDataset(
        decoder_data_path,
        tokenizer,
        max_length=getattr(args, 'decoder_max_len', 16),
    )

    loader = DataLoader(
        decoder_dataset,
        batch_size=getattr(args, 'decoder_batch_size', 32),
        shuffle=True,
        num_workers=0,  # Colab-safe
        pin_memory=device.type == "cuda",
    )

    # Freeze main model
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze decoder only
    for param in model.decoder.parameters():
        param.requires_grad = True

    # Optimizer
    decoder_lr = getattr(args, 'decoder_lr', 1e-4)
    optimizer = AdamW(
        model.decoder.parameters(),
        lr=decoder_lr,
        weight_decay=0.01,
    )

    # Scheduler
    num_steps = len(loader) * getattr(args, 'decoder_epochs', 5)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(num_steps * 0.1),
        num_training_steps=num_steps,
    )

    # Gather all label embeddings for negatives (detached — no grad to backbone during decoder training)
    all_labels = torch.cat([
        model.ner_label_embeddings,
        model.re_label_embeddings,
        model.el_label_embeddings,
    ], dim=0).detach()

    # Get label texts for true label embeddings
    # We'll use the first label from each task as a placeholder
    # In practice, you'd want to pass the true label embeddings from the dataset
    lambda_emb = getattr(args, 'lambda_emb', 1.0)
    lambda_text = getattr(args, 'lambda_text', 1.0)

    print(f"[DECODER] Training for {getattr(args, 'decoder_epochs', 5)} epochs...")
    print(f"[DECODER] LR: {decoder_lr}, Lambda_emb: {lambda_emb}, Lambda_text: {lambda_text}")

    for epoch in range(getattr(args, 'decoder_epochs', 5)):
        model.decoder.train()
        epoch_loss = []
        epoch_emb_loss = []
        epoch_text_loss = []

        for batch in loader:
            if batch is None:
                continue

            span_emb = batch['span_emb'].to(device)
            task_id = batch['task_id'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            # Sample negative labels
            neg_idx = torch.randint(0, len(all_labels), (len(span_emb), 32))
            negative_labels = all_labels[neg_idx]

            # For true label embeddings, we use a proxy: the average embedding for each task
            # Detached — no grad to backbone during decoder training
            true_label_embs = torch.stack([
                model.ner_label_embeddings[0] if tid == 0 else
                model.re_label_embeddings[0] if tid == 1 else
                model.el_label_embeddings[0] if len(model.el_label_embeddings) > 0 else model.ner_label_embeddings[0]
                for tid in task_id
            ]).detach()

            optimizer.zero_grad()

            # Forward
            generated_emb, text_logits = model.decoder(span_emb, task_id)

            # Loss
            loss, emb_loss, text_loss = combined_decoder_loss(
                generated_emb, true_label_embs, negative_labels,
                text_logits, input_ids, attention_mask,
                lambda_emb=lambda_emb,
                lambda_text=lambda_text,
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.decoder.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            epoch_loss.append(loss.item())
            epoch_emb_loss.append(emb_loss.item())
            epoch_text_loss.append(text_loss.item())

        avg_loss = np.mean(epoch_loss)
        avg_emb = np.mean(epoch_emb_loss)
        avg_text = np.mean(epoch_text_loss)
        print(f"[DECODER] Epoch {epoch}: loss={avg_loss:.4f}, emb={avg_emb:.4f}, text={avg_text:.4f}")

    # Freeze decoder after training
    for param in model.decoder.parameters():
        param.requires_grad = False

    print("[DECODER] Training complete")

    return {
        'final_loss': avg_loss,
        'final_emb_loss': avg_emb,
        'final_text_loss': avg_text,
    }


class UnifiedDataset(Dataset):
    def __init__(self, raw, tokenizer, mode="NER", max_length=256, sample_size=2000):
        self.data = []
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.mode = mode
        self.sample_size = sample_size
        self.label2idx = {}  # Label to index mapping for learnable embeddings
        self.idx2label = {}  # Index to label mapping

        if mode == "NER": self._build_ner(raw)
        elif mode == "RE": self._build_re(raw)
        elif mode == "EL": self._build_el(raw)
        else: self._build_re(raw)  # Default fallback

        # Build label vocabulary after collecting all data
        self._build_label_vocab()

        print(f"[UnifiedDataset] Successfully initialized {mode} dataset with {len(self.data)} samples.")
        print(f"[UnifiedDataset] Label vocabulary: {self.label2idx}")
        sys.stdout.flush()

    def _build_label_vocab(self):
        """Build label2idx and idx2label mappings from collected data."""
        unique_labels = sorted(set([item['label'] for item in self.data]))
        self.label2idx = {label: idx for idx, label in enumerate(unique_labels)}
        self.idx2label = {idx: label for label, idx in self.label2idx.items()}

        # Replace label strings with indices in data
        for item in self.data:
            item['label_idx'] = self.label2idx[item['label']]

    def get_label2idx(self):
        """Return the label to index mapping."""
        return self.label2idx

    def _build_ner(self, raw):
        sys.stdout.flush()
        samples_found = 0
        max_samples = self.sample_size
        for i, item in enumerate(raw):
            # Early exit if we've found enough samples
            if samples_found >= max_samples:
                print(f"[_build_ner] Found {max_samples} samples, stopping early at item {i}/{len(raw)}")
                sys.stdout.flush()
                break

            if i == 0:
                # Debug: print first item structure
                print(f"[_build_ner] First item keys: {list(item.keys())[:10]}")
                if 'conversations' in item:
                    convs = item['conversations']
                    print(f"[_build_ner] conversations length: {len(convs)}")
                    for j, turn in enumerate(convs):
                        role = turn.get('from', '?')
                        val = turn.get('value', '')[:300]
                        print(f"[_build_ner]   Turn {j} ({role}): {val}...")
                if 'entity_type' in item:
                    print(f"[_build_ner] entity_type field: {item['entity_type']}")
                if 'text' in item:
                    print(f"[_build_ner] text field (first 200 chars): {item['text'][:200]}...")
                if 'entities' in item:
                    print(f"[_build_ner] entities field: {item['entities'][:3] if isinstance(item['entities'], list) else item['entities']}")
                sys.stdout.flush()

            if i % 50 == 0 and i > 0:
                print(f"[_build_ner] processed {i}/{len(raw)} items, {len(self.data)} samples")
                sys.stdout.flush()
            try:
                # Universal-NER format: has 'conversations' with 'from' and 'value' fields
                # Also has 'text' field and 'entities' field with 'entity' and 'type'
                text = item.get('text', '')
                entities = item.get('entities', [])

                if text and entities:
                    # This is the simpler format with direct text + entities
                    for ent in entities[:3]:  # Limit to 3 entities per sample
                        entity_text = ent.get('entity', '') if isinstance(ent, dict) else ent
                        entity_type = ent.get('type', 'entity') if isinstance(ent, dict) else 'entity'

                        if not entity_text or entity_text not in text:
                            continue

                        enc = self.tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=self.max_length)
                        offsets = enc['offset_mapping']

                        s_char = text.find(entity_text)
                        if s_char == -1: continue
                        s_tok, e_tok = char_to_token_span(offsets, s_char, s_char + len(entity_text))

                        # Add task prefix to label
                        prefixed_label = f"{NER_PREFIX}{entity_type}"

                        self.data.append({
                            'input_ids': torch.tensor(enc['input_ids']),
                            'attention_mask': torch.tensor(enc['attention_mask']),
                            's_start': s_tok, 's_end': e_tok,
                            'label': prefixed_label  # Prefixed label
                        })
                        samples_found += 1
                    continue

                # Fallback to conversations format (Universal-NER style)
                convs = item.get('conversations', [])
                entity_type = item.get('entity_type', 'entity')  # Check for stored entity type

                if not convs:
                    continue

                # Universal-NER conversation format:
                # Turn 0 (human): Text: <text>
                # Turn 1 (gpt): I've read this text.
                # Turn 2 (human): What describes <entity_type> in the text?
                # Turn 3 (gpt): ["entity1", "entity2", ...]
                # ... repeated for different entity types

                human_text = ""
                # Extract the source text from the first human turn
                for turn in convs:
                    role = turn.get('from', '').lower()
                    val = turn.get('value', '')
                    if role in ['human', 'user'] and 'Text:' in val:
                        human_text = val.split('Text:')[-1].strip()
                        break

                if not human_text:
                    continue

                # Parse all (question, answer) pairs for entities
                # The conversation alternates: human asks "What describes X?", gpt answers with JSON
                current_entity_type = None
                entity_type_pairs = []

                for j, turn in enumerate(convs):
                    role = turn.get('from', '').lower()
                    val = turn.get('value', '')

                    if role in ['human', 'user']:
                        # Check if this is a "What describes X in the text?" question
                        match = re.search(r'What describes (\w+(?: \w+)*) in the text', val)
                        if match:
                            current_entity_type = match.group(1).lower()
                        elif 'What describes' in val and 'in the text' in val:
                            # Try to extract the entity type more flexibly
                            match = re.search(r'What describes (.+?) in the text', val)
                            if match:
                                current_entity_type = match.group(1).strip().lower()

                    elif role in ['gpt', 'chatgpt', 'assistant'] and current_entity_type:
                        # This should be a JSON response with entities of the current type
                        if val.strip().startswith('[') and val.strip().endswith(']'):
                            try:
                                entities = json.loads(val)
                                if isinstance(entities, list):
                                    for ent in entities:
                                        if isinstance(ent, str) and ent in human_text:
                                            entity_type_pairs.append((ent, current_entity_type))
                            except:
                                pass

                        # Reset for next pair
                        current_entity_type = None

                # Debug: log parsing results
                if i < 3:
                    print(f"[_build_ner] Item {i}: Extracted {len(entity_type_pairs)} entity pairs from conversation")
                    for j, (ent, typ) in enumerate(entity_type_pairs[:5]):
                        print(f"  '{ent}' -> {typ}")

                if not entity_type_pairs:
                    continue

                # Normalize entity types - map Universal-NER types to standard NER types
                type_mapping = {
                    # Person types
                    'person': 'PER', 'people': 'PER', 'human': 'PER', 'character': 'PER',
                    # Location types
                    'location': 'LOC', 'place': 'LOC', 'city': 'LOC', 'country': 'LOC',
                    # Organization types
                    'organization': 'ORG', 'company': 'ORG', 'org': 'ORG', 'product': 'ORG',
                    # Misc
                    'miscellaneous': 'MISC', 'misc': 'MISC', 'thing': 'MISC',
                    # Universal-NER specific - map to MISC for now
                    'programming concept': 'MISC', 'programming language': 'MISC',
                    'database': 'MISC', 'variable': 'MISC', 'date': 'MISC', 'time': 'MISC',
                    'event': 'MISC', 'concept': 'MISC', 'technology': 'MISC',
                    'group': 'ORG', 'research': 'MISC', 'module': 'MISC',
                    'unit': 'MISC', 'feature': 'MISC', 'cell': 'MISC',
                    'equipment': 'MISC', 'medical condition': 'MISC',
                    'measurement': 'MISC', 'time duration': 'MISC',
                    'treatment': 'MISC', 'symptom': 'MISC',
                }

                valid_entity_pairs = []
                for ent, typ in entity_type_pairs:
                    ent = ent.strip()
                    typ = typ.strip().lower()
                    # Map to standard NER types
                    typ = type_mapping.get(typ, typ.upper())
                    # Default to MISC for unknown types
                    if typ not in ['PER', 'LOC', 'ORG', 'MISC', 'TIME', 'NUM', 'DATE']:
                        typ = 'MISC'
                    if len(ent) > 1 and ent in human_text:
                        valid_entity_pairs.append((ent, typ))

                # Debug: log why no valid entities
                if i < 5 and not valid_entity_pairs:
                    print(f"[_build_ner] Item {i}: No valid entity pairs extracted from conversation")

                if not valid_entity_pairs:
                    continue

                # Truncate text early to avoid slow tokenization
                if len(human_text) > self.max_length * 4:
                    human_text = human_text[:self.max_length * 4]

                enc = self.tokenizer(human_text, return_offsets_mapping=True, truncation=True, max_length=self.max_length)
                offsets = enc['offset_mapping']

                # Use entity-type pairs (unique entities, keep first occurrence of type)
                seen_entities = set()
                for ent, typ in valid_entity_pairs:
                    if ent in seen_entities:
                        continue
                    seen_entities.add(ent)

                    s_char = human_text.find(ent)
                    if s_char == -1: continue
                    s_tok, e_tok = char_to_token_span(offsets, s_char, s_char + len(ent))

                    self.data.append({
                        'input_ids': torch.tensor(enc['input_ids']),
                        'attention_mask': torch.tensor(enc['attention_mask']),
                        's_start': s_tok, 's_end': e_tok,
                        'label': f"{NER_PREFIX}{typ}"  # Add task prefix
                    })
                    samples_found += 1
                    if samples_found >= max_samples:
                        break
            except Exception as e:
                # Debug: log first few errors to understand data format
                print(f"[_build_ner] Error at item {i}: {e}")
                sys.stdout.flush()
                continue

    def _build_re(self, raw):
        print(f"[_build_re] starting with {len(raw)} raw items")
        sys.stdout.flush()
        samples_found = 0
        max_samples = self.sample_size
        for i, item in enumerate(raw):
            # Early exit if we've found enough samples
            if samples_found >= max_samples:
                print(f"[_build_re] Found {max_samples} samples, stopping early at item {i}/{len(raw)}")
                sys.stdout.flush()
                break

            if i % 50 == 0 and i > 0:
                print(f"[_build_re] processed {i}/{len(raw)} items, {len(self.data)} samples")
                sys.stdout.flush()
            try:
                sents = item.get('sents', [])
                if not sents: continue

                text = " ".join([" ".join(s) for s in sents])
                labels = item.get('labels', [])
                vertex_set = item.get('vertexSet', [])

                if not labels or not vertex_set: continue

                enc = self.tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=self.max_length)

                for rel in labels[:2]:
                    h_ent = vertex_set[rel['h']][0]['name']
                    t_ent = vertex_set[rel['t']][0]['name']
                    # Use actual relation type from rel['r'] (e.g., "P150", "located_in")
                    relation_type = rel.get('r', 'relation')

                    h_char, t_char = text.find(h_ent), text.find(t_ent)
                    if h_char == -1 or t_char == -1: continue

                    h_s, h_e = char_to_token_span(enc['offset_mapping'], h_char, h_char + len(h_ent))
                    t_s, t_e = char_to_token_span(enc['offset_mapping'], t_char, t_char + len(t_ent))

                    self.data.append({
                        'input_ids': torch.tensor(enc['input_ids']),
                        'attention_mask': torch.tensor(enc['attention_mask']),
                        's_start': h_s, 's_end': h_e,
                        'o_start': t_s, 'o_end': t_e,
                        'label': f"{REL_PREFIX}{relation_type}"  # Add task prefix
                    })
                    samples_found += 1
            except Exception as e:
                if i < 5:  # Only print first few errors
                    print(f"[_build_re] Error at item {i}: {e}")
                    sys.stdout.flush()
                continue

    def _build_el(self, raw):
        """Build Entity Linking dataset from AIDA-CoNLL or ZESHEL format.

        Format expected:
        - text: document text
        - mentions: list of {'text': str, 'kb_id': str, 'label': str}
        - kb_id: Q-ID (Wikidata) or similar KB identifier
        - label: KB entity label (e.g., "Elon Musk", "Tesla, Inc.")
        """
        print(f"[_build_el] starting with {len(raw)} raw items")
        sys.stdout.flush()
        samples_found = 0
        max_samples = self.sample_size
        for i, item in enumerate(raw):
            # Early exit if we've found enough samples
            if samples_found >= max_samples:
                print(f"[_build_el] Found {max_samples} samples, stopping early at item {i}/{len(raw)}")
                sys.stdout.flush()
                break

            if i % 50 == 0 and i > 0:
                print(f"[_build_el] processed {i}/{len(raw)} items, {len(self.data)} samples")
                sys.stdout.flush()
            try:
                # Handle different EL dataset formats
                # Format 1: Direct text + mentions
                text = item.get('text', '')
                mentions = item.get('mentions', [])

                # Format 2: AIDA-style with entity spans
                if not text:
                    # Try to reconstruct from sentences
                    sents = item.get('sents', [])
                    if sents:
                        text = " ".join([" ".join(s) for s in sents])
                        mentions = []
                        # Extract entities from vertexSet
                        vertex_set = item.get('vertexSet', [])
                        for vertex in vertex_set:
                            if vertex:
                                v = vertex[0]
                                mentions.append({
                                    'text': v.get('name', ''),
                                    'kb_id': v.get('id', ''),
                                    'label': v.get('name', '')
                                })

                if not text or not mentions:
                    continue

                enc = self.tokenizer(text, return_offsets_mapping=True,
                                    truncation=True, max_length=self.max_length)
                offsets = enc['offset_mapping']

                for mention in mentions[:5]:  # Limit mentions per doc
                    mention_text = mention.get('text', '')
                    kb_id = mention.get('kb_id', '')
                    label = mention.get('label', mention_text)

                    if not mention_text or mention_text not in text:
                        continue

                    s_char = text.find(mention_text)
                    if s_char == -1: continue
                    s_tok, e_tok = char_to_token_span(offsets, s_char,
                                                      s_char + len(mention_text))

                    # Create LINK label: [LINK] Label: description
                    # For training, we use a simplified format
                    prefixed_label = f"{LINK_PREFIX}{label}"

                    self.data.append({
                        'input_ids': torch.tensor(enc['input_ids']),
                        'attention_mask': torch.tensor(enc['attention_mask']),
                        's_start': s_tok, 's_end': e_tok,
                        'label': prefixed_label,
                        'kb_id': kb_id  # Store for inference/metadata
                    })
                    samples_found += 1

            except Exception as e:
                if i < 5:  # Only print first few errors
                    print(f"[_build_el] Error at item {i}: {e}")
                    sys.stdout.flush()
                continue

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

class CollateFn:
    """Picklable collate function for multiprocessing DataLoader."""
    def __init__(self, mode="ner"):
        self.mode = mode

    def __call__(self, batch):
        if not batch:
            return None
        out = {}
        for k in batch[0].keys():
            if k not in ['label', 'label_idx']:
                items = [x[k] for x in batch]
                if k in ['input_ids', 'attention_mask']:
                    out[k] = torch.nn.utils.rnn.pad_sequence(items, batch_first=True, padding_value=0)
                elif k == 'kb_id':
                    # Skip string fields - store as list for metadata only
                    continue
                else:
                    out[k] = torch.tensor(items)

        # Return label_indices to process in training loop
        label_indices = torch.tensor([x['label_idx'] for x in batch], dtype=torch.long)
        out['label_indices'] = label_indices
        return out

def collate_fn(batch, tokenizer, model, device, mode="ner"):
    """Legacy function for backwards compatibility."""
    if not batch: return None
    out = {}
    for k in batch[0].keys():
        if k not in ['label', 'label_idx']:
            items = [x[k] for x in batch]
            if k in ['input_ids', 'attention_mask']:
                out[k] = torch.nn.utils.rnn.pad_sequence(items, batch_first=True, padding_value=0)
            elif k == 'kb_id':
                # Skip string fields
                continue
            else:
                out[k] = torch.tensor(items)

    # Use learnable label embeddings - return label_indices to process in training loop
    # (Cannot use CUDA in worker process)
    label_indices = torch.tensor([x['label_idx'] for x in batch], dtype=torch.long)
    out['label_indices'] = label_indices
    return out

def evaluate_zero_shot(model, dataloader, device, seen_labels, heldout_labels,
                       mode="NER", temperature=0.07, max_batches=10):
    """Evaluate zero-shot retrieval on heldout labels.

    Computes:
    1. Seen label similarity (labels used in training)
    2. Heldout label similarity (labels never seen during training)
    3. Zero-shot gap: difference between seen and heldout similarity

    Args:
        model: UnifiedKGModel
        dataloader: DataLoader for validation data
        device: torch device
        seen_labels: Set of label names (with prefix) used during training
        heldout_labels: Set of label names (with prefix) held out for testing
        mode: "NER" or "RE"
        temperature: Temperature for similarity scaling
        max_batches: Max batches to evaluate (for speed)

    Returns:
        metrics dict with seen_sim, heldout_sim, zero_shot_gap
    """
    model.eval()

    # Get label embeddings and indices
    if mode == "NER":
        all_label_embeddings = model.ner_label_embeddings
        label2idx = model.ner_label2idx
    elif mode == "RE":
        all_label_embeddings = model.re_label_embeddings
        label2idx = model.re_label2idx
    else:  # EL
        all_label_embeddings = model.el_label_embeddings
        label2idx = model.el_label2idx

    # Get indices for seen and heldout labels
    seen_indices = [label2idx[l] for l in seen_labels if l in label2idx]
    heldout_indices = [label2idx[l] for l in heldout_labels if l in label2idx]

    if not heldout_indices:
        # No heldout labels in this dataset
        return {"seen_sim": 0.0, "heldout_sim": 0.0, "zero_shot_gap": 0.0}

    # Collect similarities for true labels
    seen_sims = []
    heldout_sims = []

    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            if i >= max_batches:
                break
            if batch is None:
                continue

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            label_indices = batch['label_indices'].to(device)

            # Get hidden states
            hidden = model.backbone(input_ids, attention_mask).last_hidden_state

            # Get predictions based on mode
            if mode == "NER":
                preds = model.encode_spans_ner(hidden, batch['s_start'], batch['s_end'])
            elif mode == "RE":
                preds = model.encode_pairs_re(hidden, batch['s_start'], batch['s_end'],
                                               batch['o_start'], batch['o_end'])
            else:  # EL
                preds = model.encode_mentions_el(hidden, batch['s_start'], batch['s_end'])

            # Compute cosine similarity in chunks to avoid OOM on large label sets
            label_chunk_size = 512
            num_labels = all_label_embeddings.shape[0]
            batch_size = preds.shape[0]
            arange_b = torch.arange(batch_size, device=device)

            # Gather the similarity for each sample's true label
            true_sim = torch.zeros(batch_size, device=device)
            # Process labels in chunks to keep memory bounded
            for chunk_start in range(0, num_labels, label_chunk_size):
                chunk_end = min(chunk_start + label_chunk_size, num_labels)
                chunk_emb = all_label_embeddings[chunk_start:chunk_end]  # (C, D)
                chunk_sim = F.cosine_similarity(
                    preds.unsqueeze(1),  # (B, 1, D)
                    chunk_emb.unsqueeze(0),  # (1, C, D)
                    dim=-1
                )  # (B, C)

                # Check which samples have their true label in this chunk
                in_chunk = (label_indices >= chunk_start) & (label_indices < chunk_end)
                if in_chunk.any():
                    local_idx = label_indices[in_chunk] - chunk_start
                    true_sim[in_chunk] = chunk_sim[arange_b[in_chunk], local_idx]

            # Track by label type
            for idx, sim in zip(label_indices.cpu().numpy(), true_sim.cpu().numpy()):
                if idx in seen_indices:
                    seen_sims.append(sim)
                elif idx in heldout_indices:
                    heldout_sims.append(sim)

    # Compute metrics
    seen_sim = sum(seen_sims) / len(seen_sims) if seen_sims else 0.0
    heldout_sim = sum(heldout_sims) / len(heldout_sims) if heldout_sims else 0.0
    zero_shot_gap = seen_sim - heldout_sim

    model.train()

    return {
        "seen_sim": seen_sim,
        "heldout_sim": heldout_sim,
        "zero_shot_gap": zero_shot_gap,
        "seen_count": len(seen_sims),
        "heldout_count": len(heldout_sims)
    }


def split_dataset_indices(dataset_length, val_frac=0.1, test_frac=0.1, seed=42):
    """Split dataset indices into train/val/test using deterministic random.

    Args:
        dataset_length: Total number of samples in the dataset.
        val_frac: Fraction of data for validation (default 0.1).
        test_frac: Fraction of data for test (default 0.1).
        seed: Random seed for reproducibility.

    Returns:
        Tuple of (train_indices, val_indices, test_indices) as lists.
    """
    if val_frac == 0 and test_frac == 0:
        return list(range(dataset_length)), [], []

    import random as _random
    rng = _random.Random(seed)

    indices = list(range(dataset_length))
    rng.shuffle(indices)

    n_test = int(dataset_length * test_frac)
    n_val = int(dataset_length * val_frac)

    # Guard against empty splits for small datasets
    if dataset_length > 5:
        n_test = max(n_test, 1) if test_frac > 0 else 0
        n_val = max(n_val, 1) if val_frac > 0 else 0
    else:
        # Very small dataset: don't split
        return list(range(dataset_length)), [], []

    test_indices = indices[:n_test]
    val_indices = indices[n_test:n_test + n_val]
    train_indices = indices[n_test + n_val:]

    # Guard: if train is empty, put everything in train
    if not train_indices:
        train_indices = list(range(dataset_length))
        val_indices = []
        test_indices = []

    return train_indices, val_indices, test_indices


def evaluate_val_loss(model, ner_val_loader, re_val_loader, el_val_loader,
                      device, args, zero_shot_eval, seen_ner_indices,
                      seen_re_indices, max_batches=20):
    """Evaluate model on validation loaders, mirroring the training loss computation.

    Returns dict with val_loss, val_ner_sim, val_re_sim, val_el_sim.
    """
    model.eval()
    total_loss = 0.0
    total_ner_sim = 0.0
    total_re_sim = 0.0
    total_el_sim = 0.0
    n_batches = 0

    with torch.no_grad():
        # Evaluate NER val loss
        if ner_val_loader is not None:
            ner_losses = []
            ner_sims = []
            for i, batch in enumerate(ner_val_loader):
                if i >= max_batches:
                    break
                if batch is None:
                    continue
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                label_indices = batch['label_indices'].to(device)
                hidden = model.backbone(input_ids, attention_mask).last_hidden_state
                preds = model.encode_spans_ner(hidden, batch['s_start'], batch['s_end'])

                if zero_shot_eval and len(seen_ner_indices) < len(model.ner_label_embeddings):
                    loss, sim = cosine_cross_entropy_loss_zero_shot(
                        preds, model.ner_label_embeddings, label_indices,
                        seen_indices=None, temperature=model.temperature,
                        contrastive_weight=0.05
                    )
                else:
                    use_arcface = getattr(args, 'use_arcface', True)
                    if use_arcface:
                        loss, sim = arcface_loss(
                            preds, model.ner_label_embeddings, label_indices,
                            temperature=model.temperature,
                            margin_ner=0.05, margin_re=0.0, task_type="NER"
                        )
                    else:
                        loss, sim = cosine_cross_entropy_loss(
                            preds, model.ner_label_embeddings, label_indices,
                            temperature=model.temperature
                        )
                ner_losses.append(loss.item())
                ner_sims.append(sim)
            if ner_losses:
                total_loss += sum(ner_losses) / len(ner_losses)
                total_ner_sim = sum(ner_sims) / len(ner_sims)
                n_batches += 1

        # Evaluate RE val loss
        if re_val_loader is not None:
            re_losses = []
            re_sims = []
            for i, batch in enumerate(re_val_loader):
                if i >= max_batches:
                    break
                if batch is None:
                    continue
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                label_indices = batch['label_indices'].to(device)
                hidden = model.backbone(input_ids, attention_mask).last_hidden_state
                preds = model.encode_pairs_re(hidden, batch['s_start'], batch['s_end'],
                                               batch['o_start'], batch['o_end'])

                if zero_shot_eval and len(seen_re_indices) < len(model.re_label_embeddings):
                    loss, sim = cosine_cross_entropy_loss_zero_shot(
                        preds, model.re_label_embeddings, label_indices,
                        seen_indices=None, temperature=model.temperature,
                        contrastive_weight=0.05
                    )
                else:
                    use_arcface = getattr(args, 'use_arcface', True)
                    if use_arcface:
                        loss, sim = arcface_loss(
                            preds, model.re_label_embeddings, label_indices,
                            temperature=model.temperature,
                            margin_ner=0.0, margin_re=0.0, task_type="RE"
                        )
                    else:
                        loss, sim = cosine_cross_entropy_loss(
                            preds, model.re_label_embeddings, label_indices,
                            temperature=model.temperature
                        )
                re_losses.append(loss.item())
                re_sims.append(sim)
            if re_losses:
                total_loss += sum(re_losses) / len(re_losses)
                total_re_sim = sum(re_sims) / len(re_sims)
                n_batches += 1

        # Evaluate EL val loss
        if el_val_loader is not None:
            el_losses = []
            el_sims = []
            for i, batch in enumerate(el_val_loader):
                if i >= max_batches:
                    break
                if batch is None:
                    continue
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                label_indices = batch['label_indices'].to(device)
                hidden = model.backbone(input_ids, attention_mask).last_hidden_state
                preds = model.encode_mentions_el(hidden, batch['s_start'], batch['s_end'])

                if model.el_label_embeddings.shape[0] > 0:
                    loss, sim = cosine_cross_entropy_loss_chunked(
                        preds, model.el_label_embeddings, label_indices,
                        temperature=model.temperature, chunk_size=1024
                    )
                    el_losses.append(loss.item())
                    el_sims.append(sim)
            if el_losses:
                total_loss += sum(el_losses) / len(el_losses)
                total_el_sim = sum(el_sims) / len(el_sims)
                n_batches += 1

    model.train()

    avg_loss = total_loss / n_batches if n_batches > 0 else 0.0
    return {
        'val_loss': avg_loss,
        'val_ner_sim': total_ner_sim,
        'val_re_sim': total_re_sim,
        'val_el_sim': total_el_sim,
    }


def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    logger.info("Loading streaming datasets...")
    logger.info("Using DIVERSE data sources for better generalization")

    # === DIVERSE NER DATASETS ===
    # Multiple sources for robust NER training
    ner_raw = []
    ner_sources = [
        ("conll2003", 30000, "train"),
        ("tomaarsen/ontonotes", 30000, "train"),
        ("Universal-NER/Pile-NER-type", 20000, "train"),
        ("wikiann", "wikiann", "en", 20000),  # Multilingual NER
    ]

    tag_map = {
        'B-PER': 'PER', 'I-PER': 'PER',
        'B-ORG': 'ORG', 'I-ORG': 'ORG',
        'B-LOC': 'LOC', 'I-LOC': 'LOC',
        'B-MISC': 'MISC', 'I-MISC': 'MISC',
        # Additional tags from OntoNotes
        'B-CARDINAL': 'MISC', 'I-CARDINAL': 'MISC',
        'B-DATE': 'DATE', 'I-DATE': 'DATE',
        'B-EVENT': 'MISC', 'I-EVENT': 'MISC',
        'B-FAC': 'ORG', 'I-FAC': 'ORG',
        'B-GPE': 'LOC', 'I-GPE': 'LOC',
        'B-LAW': 'MISC', 'I-LAW': 'MISC',
        'B-LOC': 'LOC', 'I-LOC': 'LOC',
        'B-MONEY': 'MISC', 'I-MONEY': 'MISC',
        'B-NORP': 'ORG', 'I-NORP': 'ORG',
        'B-ORDINAL': 'MISC', 'I-ORDINAL': 'MISC',
        'B-ORG': 'ORG', 'I-ORG': 'ORG',
        'B-PERCENT': 'MISC', 'I-PERCENT': 'MISC',
        'B-PERSON': 'PER', 'I-PERSON': 'PER',
        'B-PRODUCT': 'MISC', 'I-PRODUCT': 'MISC',
        'B-QUANTITY': 'MISC', 'I-QUANTITY': 'MISC',
        'B-TIME': 'TIME', 'I-TIME': 'TIME',
        'B-WORK_OF_ART': 'MISC', 'I-WORK_OF_ART': 'MISC',
    }

    # Load from multiple NER sources
    for source_info in ner_sources:
        if len(ner_raw) >= args.sample_size:
            break

        if len(source_info) == 4:  # wikiann format
            dataset_name, subset, lang, target_size = source_info
            source_name = f"{dataset_name}/{subset}/{lang}"
        else:
            dataset_name, target_size, split = source_info
            source_name = dataset_name

        try:
            print(f"[DATA] Loading NER from {source_name} (target {target_size})...")
            if len(source_info) == 4:
                ds = load_dataset(dataset_name, subset, split=split, streaming=True)
            else:
                _kwargs = dict(split=split, streaming=True)
                if dataset_name == "conll2003":
                    _kwargs["revision"] = "refs/convert/parquet"
                ds = load_dataset(dataset_name, **_kwargs)

            for item in ds:
                if len(ner_raw) >= args.sample_size:
                    break

                # Handle different dataset formats
                if dataset_name == "conll2003":
                    tokens = item['tokens']
                    ner_tags = item['ner_tags']
                    text = " ".join(tokens)

                    # Find entity spans
                    i = 0
                    while i < len(tokens):
                        tag = ner_tags[i]
                        if tag >= 0:  # CoNLL uses numeric tags
                            tag_str = f"B-{['PER', 'ORG', 'LOC', 'MISC'][tag]}" if tag < 4 else "B-MISC"
                        else:
                            tag_str = 'O'
                        if isinstance(tag, int):
                            # Map numeric tag to string
                            tag_to_str = {0: 'B-PER', 1: 'I-PER', 2: 'B-ORG', 3: 'I-ORG',
                                          4: 'B-LOC', 5: 'I-LOC', 6: 'B-MISC', 7: 'I-MISC'}
                            tag_str = tag_to_str.get(tag, 'O')

                        # Re-parse with proper string tags
                        i = 0
                        while i < len(tokens):
                            tag_val = ner_tags[i]
                            tag_str = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'][tag_val] if tag_val < 9 else 'O'

                            if tag_str.startswith('B-'):
                                ent_type = tag_map.get(tag_str, 'MISC')
                                ent_tokens = [tokens[i]]
                                j = i + 1
                                while j < len(tokens) and ner_tags[j] < 9:
                                    next_tag = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'][ner_tags[j]]
                                    if next_tag.startswith('I-') and tag_map.get(next_tag, 'MISC') == ent_type:
                                        ent_tokens.append(tokens[j])
                                        j += 1
                                    else:
                                        break
                                ent_text = " ".join(ent_tokens)
                                ner_raw.append({
                                    'text': text,
                                    'entities': [{'entity': ent_text, 'type': ent_type}]
                                })
                                i = j
                            else:
                                i += 1
                        break  # Process one item per iteration

                elif dataset_name == "tomaarsen/ontonotes":
                    tokens = item['tokens']
                    ner_tags = item['ner_tags']
                    text = " ".join(tokens)

                    # OntoNotes uses numeric tags
                    for i, tag_val in enumerate(ner_tags):
                        if tag_val > 0:  # Non-O tag
                            # Find full entity span
                            j = i
                            while j < len(ner_tags) and ner_tags[j] == tag_val:
                                j += 1
                            ent_text = " ".join(tokens[i:j])
                            # Map to standard types
                            ent_type = 'PER'  # Simplified - could map properly
                            ner_raw.append({
                                'text': text,
                                'entities': [{'entity': ent_text, 'type': ent_type}]
                            })
                            i = j
                    if len(ner_raw) % 1000 == 0:
                        print(f"[DATA] Loaded {len(ner_raw)} NER samples so far...")

                elif dataset_name == "Universal-NER/Pile-NER-type":
                    # Keep as-is for UnifiedDataset to parse
                    ner_raw.append(item)

                elif dataset_name == "wikiann":
                    tokens = item['tokens']
                    ner_tags = item['ner_tags']
                    text = " ".join(tokens)

                    # WikiANN uses BIO tags
                    i = 0
                    while i < len(tokens):
                        tag_str = ner_tags[i]
                        if tag_str.startswith('B-'):
                            ent_type = tag_str[2:]  # Remove 'B-'
                            # Map to standard types
                            if ent_type not in ['PER', 'ORG', 'LOC']:
                                ent_type = 'MISC'
                            ent_tokens = [tokens[i]]
                            j = i + 1
                            while j < len(tokens) and ner_tags[j].startswith('I-'):
                                ent_tokens.append(tokens[j])
                                j += 1
                            ent_text = " ".join(ent_tokens)
                            ner_raw.append({
                                'text': text,
                                'entities': [{'entity': ent_text, 'type': ent_type}]
                            })
                            i = j
                        else:
                            i += 1

            print(f"[DATA] Loaded {len(ner_raw)} NER samples from {source_name}")

        except Exception as e:
            print(f"[DATA] Failed to load {source_name}: {e}")
            continue

    print(f"[DATA] Total NER samples: {len(ner_raw)}")

    # === DIVERSE RE DATASETS ===
    # Multiple sources for robust RE training
    re_raw = []

    # Load from multiple RE sources
    re_sources = [
        ("tonytan48/Re-DocRED", "train", 40000),
        ("docred", "train_annotated", 20000),  # Original DocRED
    ]

    for dataset_name, split, target_size in re_sources:
        if len(re_raw) >= args.sample_size * 2:  # More RE data
            break

        try:
            print(f"[DATA] Loading RE from {dataset_name} (target {target_size})...")
            ds = load_dataset(dataset_name, split=split, streaming=True)

            count = 0
            for item in ds:
                if len(re_raw) >= args.sample_size * 2:
                    break

                re_raw.append(item)
                count += 1

            print(f"[DATA] Loaded {count} samples from {dataset_name}")

        except Exception as e:
            print(f"[DATA] Failed to load {dataset_name}: {e}")
            # Fallback to just Re-DocRED
            if not re_raw:
                print(f"[DATA] Using fallback Re-DocRED...")
                re_raw = list(load_dataset("tonytan48/Re-DocRED", split="train", streaming=True).take(args.sample_size * 50))
            continue

    print(f"[DATA] Total RE samples: {len(re_raw)}")

    # === ENTITY LINKING DATASETS ===
    # Load from EL sources (AIDA-CoNLL, etc.)
    el_raw = []

    # Check if EL is enabled (default: True)
    use_el = getattr(args, 'use_el', True)
    el_sample_size = getattr(args, 'el_sample_size', 10000)

    if use_el:
        el_sources = [
            ("saehri/aida_conll", "train", el_sample_size),  # AIDA-CoNLL for EL
        ]

        for dataset_name, split, target_size in el_sources:
            if len(el_raw) >= el_sample_size:
                break

            try:
                print(f"[DATA] Loading EL from {dataset_name} (target {target_size})...")
                ds = load_dataset(dataset_name, split=split, streaming=True)

                count = 0
                for item in ds:
                    if len(el_raw) >= el_sample_size:
                        break

                    el_raw.append(item)
                    count += 1

                print(f"[DATA] Loaded {count} samples from {dataset_name}")

            except Exception as e:
                print(f"[DATA] Failed to load {dataset_name}: {e}")
                # Create synthetic EL data from RE dataset as fallback
                if not el_raw and len(re_raw) > 0:
                    print(f"[DATA] Creating synthetic EL data from RE dataset...")
                    for item in re_raw[:5000]:  # Limit synthetic samples
                        vertex_set = item.get('vertexSet', [])
                        sents = item.get('sents', [])
                        if not vertex_set or not sents:
                            continue

                        text = " ".join([" ".join(s) for s in sents])

                        mentions = []
                        for vertex in vertex_set[:5]:  # Up to 5 entities per doc
                            if vertex:
                                v = vertex[0]
                                entity_name = v.get('name', '')
                                if entity_name and entity_name in text:
                                    mentions.append({
                                        'text': entity_name,
                                        'kb_id': v.get('id', 'UNKNOWN'),
                                        'label': entity_name
                                    })

                        if mentions:
                            el_raw.append({
                                'text': text,
                                'mentions': mentions
                            })

                            if len(el_raw) >= el_sample_size:
                                break
                continue

        print(f"[DATA] Total EL samples: {len(el_raw)}")

    ner_ds = UnifiedDataset(ner_raw, tokenizer, "NER", max_length=args.max_length, sample_size=args.sample_size)
    re_ds = UnifiedDataset(re_raw, tokenizer, "RE", max_length=args.max_length, sample_size=args.sample_size)

    # Only create EL dataset if we have data
    el_ds = None
    if el_raw:
        el_ds = UnifiedDataset(el_raw, tokenizer, "EL", max_length=args.max_length, sample_size=el_sample_size)
        print(f"[DATA] EL dataset created with {len(el_ds)} samples")
    else:
        print(f"[DATA] No EL data available, skipping EL task")

    if len(ner_ds) == 0:
        logger.warning("NER dataset empty. Attempting backup dataset (OntoNotes)...")
        # Use OntoNotes which works with newer datasets library
        try:
            onto = load_dataset("tomaarsen/ontonotes", split="train", streaming=True)
            ner_raw_backup = []
            for x in onto:
                if len(ner_raw_backup) >= 500: break
                text = " ".join(x['tokens'])
                # Mock structure to pass to _build_ner
                for i, label in enumerate(x['ner_tags']):
                    if label > 0:
                        ner_raw_backup.append({
                            'conversations': [
                                {'from': 'human', 'value': f"Text: {text}"},
                                {'from': 'gpt', 'value': json.dumps([x['tokens'][i]])}
                            ]
                        })
            ner_ds = UnifiedDataset(ner_raw_backup, tokenizer, "NER", max_length=args.max_length, sample_size=args.sample_size)
        except Exception as e:
            logger.error(f"Backup dataset also failed: {e}")

    # Final fallback: create better synthetic NER data from entities found in RE data
    if len(ner_ds) == 0 and len(re_ds) > 0:
        logger.warning("Creating synthetic NER dataset from RE vertexSet...")
        sys.stdout.flush()
        print("[DEBUG] Creating synthetic NER data from RE vertexSet...")
        sys.stdout.flush()
        ner_raw_synthetic = []

        # Extract entities from RE dataset's vertexSet
        for item in re_raw:
            try:
                sents = item.get('sents', [])
                vertex_set = item.get('vertexSet', [])
                if not sents or not vertex_set:
                    continue

                # Reconstruct text from sentences
                text = " ".join([" ".join(s) for s in sents])

                # Extract unique entities with their types
                entities_seen = set()
                entities_for_text = []

                for vertex in vertex_set:
                    if not vertex:
                        continue
                    v = vertex[0]  # First mention of the entity
                    entity_name = v.get('name', '')
                    entity_type = v.get('type', 'entity')

                    if entity_name and entity_name not in entities_seen and entity_name in text:
                        entities_seen.add(entity_name)
                        entities_for_text.append({
                            'text': entity_name,
                            'type': entity_type
                        })

                # Only keep items with at least 2 entities
                if len(entities_for_text) >= 2:
                    # Create training samples - use direct format with 'text' and 'entities'
                    # Parser expects entities with 'entity' and 'type' keys
                    formatted_entities = [
                        {'entity': e['text'], 'type': e['type']}
                        for e in entities_for_text[:5]  # Up to 5 entities per text
                    ]
                    ner_raw_synthetic.append({
                        'text': text,
                        'entities': formatted_entities
                    })

                    if len(ner_raw_synthetic) >= args.sample_size:  # Cap at sample_size
                        break
            except Exception as e:
                if len(ner_raw_synthetic) == 0:
                    print(f"[DEBUG] Error creating synthetic NER: {e}")
                continue

        print(f"[DEBUG] Created {len(ner_raw_synthetic)} synthetic NER samples from RE data")
        sys.stdout.flush()

        if len(ner_raw_synthetic) == 0:
            # Ultimate fallback with hardcoded examples
            print("[DEBUG] Using hardcoded fallback examples...")
            sample_data = [
                {'text': 'Apple Inc. was founded by Steve Jobs in Cupertino.',
                 'entities': [{'entity': 'Apple Inc.', 'type': 'ORG'},
                              {'entity': 'Steve Jobs', 'type': 'PER'},
                              {'entity': 'Cupertino', 'type': 'LOC'}]},
                {'text': 'Barack Obama was born in Hawaii.',
                 'entities': [{'entity': 'Barack Obama', 'type': 'PER'},
                              {'entity': 'Hawaii', 'type': 'LOC'}]},
                {'text': 'Google acquired DeepMind in 2014.',
                 'entities': [{'entity': 'Google', 'type': 'ORG'},
                              {'entity': 'DeepMind', 'type': 'ORG'}]},
                {'text': 'The Eiffel Tower is located in Paris, France.',
                 'entities': [{'entity': 'Eiffel Tower', 'type': 'LOC'},
                              {'entity': 'Paris', 'type': 'LOC'},
                              {'entity': 'France', 'type': 'LOC'}]},
                {'text': 'Microsoft was founded by Bill Gates and Paul Allen.',
                 'entities': [{'entity': 'Microsoft', 'type': 'ORG'},
                              {'entity': 'Bill Gates', 'type': 'PER'},
                              {'entity': 'Paul Allen', 'type': 'PER'}]},
            ]
            ner_raw_synthetic = sample_data * 100  # 500 total samples

        print(f"[DEBUG] About to call UnifiedDataset with {len(ner_raw_synthetic)} samples...")
        sys.stdout.flush()
        ner_ds = UnifiedDataset(ner_raw_synthetic, tokenizer, "NER", max_length=args.max_length, sample_size=args.sample_size)
        print(f"[DEBUG] UnifiedDataset created successfully with {len(ner_ds)} samples")
        sys.stdout.flush()

    if len(ner_ds) == 0 or len(re_ds) == 0:
        logger.error(f"Dataset critical error: NER={len(ner_ds)}, RE={len(re_ds)}.")
        return

    print(f"\n[READY] Datasets ready - NER: {len(ner_ds)}, RE: {len(re_ds)}, EL: {len(el_ds) if el_ds else 0}")

    # Debug: Show sample labels
    ner_labels = set([ner_ds[i]['label'] for i in range(min(100, len(ner_ds)))])
    re_labels = set([re_ds[i]['label'] for i in range(min(100, len(re_ds)))])
    print(f"[DEBUG] NER labels (sample): {list(ner_labels)[:20]}")
    print(f"[DEBUG] RE labels (sample): {list(re_labels)[:20]}")
    if el_ds:
        el_labels = set([el_ds[i]['label'] for i in range(min(100, len(el_ds)))])
        print(f"[DEBUG] EL labels (sample): {list(el_labels)[:20]}")
    print(f"[READY] Starting training: batch_size={args.batch_size}, sample_size={args.sample_size}, lr={args.lr}, loss=cosine+CE")
    sys.stdout.flush()

    # === TRAIN/VAL/TEST SPLIT ===
    val_split = getattr(args, 'val_split', 0.0)
    test_split = getattr(args, 'test_split', 0.0)
    split_seed = getattr(args, 'split_seed', 42)

    if val_split > 0 or test_split > 0:
        print(f"\n[SPLIT] Creating train/val/test splits (val={val_split}, test={test_split}, seed={split_seed})")

        ner_train_idx, ner_val_idx, ner_test_idx = split_dataset_indices(
            len(ner_ds), val_frac=val_split, test_frac=test_split, seed=split_seed)
        re_train_idx, re_val_idx, re_test_idx = split_dataset_indices(
            len(re_ds), val_frac=val_split, test_frac=test_split, seed=split_seed + 1)

        ner_train_ds = Subset(ner_ds, ner_train_idx)
        ner_val_ds = Subset(ner_ds, ner_val_idx) if ner_val_idx else None
        ner_test_ds = Subset(ner_ds, ner_test_idx) if ner_test_idx else None

        re_train_ds = Subset(re_ds, re_train_idx)
        re_val_ds = Subset(re_ds, re_val_idx) if re_val_idx else None
        re_test_ds = Subset(re_ds, re_test_idx) if re_test_idx else None

        if el_ds is not None:
            el_train_idx, el_val_idx, el_test_idx = split_dataset_indices(
                len(el_ds), val_frac=val_split, test_frac=test_split, seed=split_seed + 2)
            el_train_ds = Subset(el_ds, el_train_idx)
            el_val_ds = Subset(el_ds, el_val_idx) if el_val_idx else None
            el_test_ds = Subset(el_ds, el_test_idx) if el_test_idx else None
        else:
            el_train_ds = el_val_ds = el_test_ds = None

        print(f"[SPLIT] NER: train={len(ner_train_ds)}, val={len(ner_val_ds) if ner_val_ds else 0}, test={len(ner_test_ds) if ner_test_ds else 0}")
        print(f"[SPLIT] RE:  train={len(re_train_ds)}, val={len(re_val_ds) if re_val_ds else 0}, test={len(re_test_ds) if re_test_ds else 0}")
        if el_ds is not None:
            print(f"[SPLIT] EL:  train={len(el_train_ds)}, val={len(el_val_ds) if el_val_ds else 0}, test={len(el_test_ds) if el_test_ds else 0}")
    else:
        print("[SPLIT] No val/test split - using full datasets for training")
        ner_train_ds = ner_ds
        ner_val_ds = ner_test_ds = None
        re_train_ds = re_ds
        re_val_ds = re_test_ds = None
        el_train_ds = el_ds
        el_val_ds = el_test_ds = None
    sys.stdout.flush()
    ner_label2idx = ner_ds.get_label2idx()
    re_label2idx = re_ds.get_label2idx()
    el_label2idx = el_ds.get_label2idx() if el_ds else {}
    print(f"[DEBUG] NER label2idx: {ner_label2idx}")
    print(f"[DEBUG] RE label2idx: {re_label2idx}")
    print(f"[DEBUG] EL label2idx: {el_label2idx}")

    # Create model - with LoRA adapters for task-specific capacity
    use_lora = getattr(args, 'use_lora', True)  # Default enabled
    lora_rank = getattr(args, 'lora_rank', 8)
    lora_alpha = getattr(args, 'lora_alpha', 16)

    print(f"[DEBUG] Creating model with LoRA={use_lora} (rank={lora_rank}, alpha={lora_alpha})...")
    model = UnifiedKGModel(
        model_name=args.model_name,
        ner_label2idx=ner_label2idx,
        re_label2idx=re_label2idx,
        el_label2idx=el_label2idx,
        tokenizer=tokenizer,
        device=device,
        use_lora=use_lora,
        lora_rank=lora_rank,
        lora_alpha=lora_alpha
    ).to(device)

    # Initialize frozen label embeddings using pplx-embed backbone
    print("[INIT] Initializing frozen label embeddings...")
    model.initialize_frozen_labels(tokenizer, device)

    # === ZERO-SHOT HELDOUT LABEL SELECTION ===
    # Select labels to hold out for zero-shot testing
    disable_zero_shot = getattr(args, 'disable_zero_shot', False)
    zero_shot_eval = getattr(args, 'zero_shot_eval', True) and not disable_zero_shot

    if zero_shot_eval:
        print("\n[ZERO-SHOT] Setting up heldout labels for zero-shot testing...")

        # Get heldout label args
        heldout_ner_arg = getattr(args, 'heldout_ner_labels', "LOC")
        heldout_re_arg = getattr(args, 'heldout_re_labels', "P150,located_in,founded_by")
        heldout_el_arg = getattr(args, 'heldout_el_labels', "")
        heldout_ratio = getattr(args, 'heldout_ratio', 0.15)
        el_heldout_ratio = getattr(args, 'el_heldout_ratio', DEFAULT_HELDOUT_EL_RATIO)

        # Parse comma-separated labels
        heldout_ner_list = [l.strip() for l in heldout_ner_arg.split(",")] if heldout_ner_arg else None
        heldout_re_list = [l.strip() for l in heldout_re_arg.split(",")] if heldout_re_arg else None
        heldout_el_list = [l.strip() for l in heldout_el_arg.split(",")] if heldout_el_arg else None

        # Select heldout labels
        seen_ner_labels, heldout_ner_labels = select_heldout_labels(
            ner_label2idx, "NER", heldout_ratio, heldout_ner_list
        )
        seen_re_labels, heldout_re_labels = select_heldout_labels(
            re_label2idx, "RE", heldout_ratio, heldout_re_list
        )

        # EL heldout (only if we have EL data)
        if el_label2idx and len(el_label2idx) > 1:
            seen_el_labels, heldout_el_labels = select_heldout_labels(
                el_label2idx, "EL", el_heldout_ratio, heldout_el_list
            )
            seen_el_indices = [el_label2idx[l] for l in seen_el_labels]
        else:
            seen_el_labels = set(el_label2idx.keys())
            heldout_el_labels = set()
            seen_el_indices = list(range(len(el_label2idx)))

        # Get indices for seen labels (for filtering embeddings during training)
        seen_ner_indices = [ner_label2idx[l] for l in seen_ner_labels]
        seen_re_indices = [re_label2idx[l] for l in seen_re_labels]

        print(f"[ZERO-SHOT] Training on {len(seen_ner_labels)} seen NER labels, testing on {len(heldout_ner_labels)} heldout")
        print(f"[ZERO-SHOT] Training on {len(seen_re_labels)} seen RE labels, testing on {len(heldout_re_labels)} heldout")
        if heldout_el_labels:
            print(f"[ZERO-SHOT] Training on {len(seen_el_labels)} seen EL labels, testing on {len(heldout_el_labels)} heldout")
        print(f"[ZERO-SHOT] Heldout NER: {heldout_ner_labels}")
        print(f"[ZERO-SHOT] Heldout RE: {heldout_re_labels}")
        if heldout_el_labels:
            print(f"[ZERO-SHOT] Heldout EL: {heldout_el_labels}")
        print()
    else:
        # No zero-shot testing - use all labels for training
        seen_ner_labels = set(ner_label2idx.keys())
        heldout_ner_labels = set()
        seen_re_labels = set(re_label2idx.keys())
        heldout_re_labels = set()
        seen_el_labels = set(el_label2idx.keys())
        heldout_el_labels = set()
        seen_ner_indices = list(range(len(ner_label2idx)))
        seen_re_indices = list(range(len(re_label2idx)))
        seen_el_indices = list(range(len(el_label2idx)))
        print("[ZERO-SHOT] Disabled - training on all labels\n")

    # Freeze backbone encoder EXCEPT LoRA parameters
    # LoRA adapters and task-specific projections are trainable
    print("[ARCH] Freezing backbone encoder - training LoRA + adapters + projections")
    for name, param in model.named_parameters():
        if "backbone" in name and "lora" not in name.lower():
            param.requires_grad = False

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"[ARCH] Trainable: {trainable_params:,} / {total_params:,} params ({100*trainable_params/total_params:.1f}%)")
    print(f"[ARCH] Frozen label embeddings preserve pplx-embed semantics for zero-shot")
    sys.stdout.flush()

    # === DataLoader ===
    # Colab: num_workers=0 is faster (spawn mode overhead > benefit for small datasets)
    # For non-Colab GPU servers, try num_workers=2-4 for faster data loading
    num_workers = 0  # Disable for Colab (spawn mode too slow)
    pin_memory = device.type == "cuda"

    # Train loaders (shuffle=True)
    print("[DEBUG] Creating NER train DataLoader...")
    sys.stdout.flush()
    ner_loader = DataLoader(ner_train_ds, batch_size=args.batch_size, shuffle=True,
                            num_workers=num_workers, pin_memory=pin_memory,
                            collate_fn=CollateFn(mode="ner"))
    print("[DEBUG] NER train DataLoader created")
    sys.stdout.flush()

    print("[DEBUG] Creating RE train DataLoader...")
    sys.stdout.flush()
    re_loader = DataLoader(re_train_ds, batch_size=args.batch_size, shuffle=True,
                           num_workers=num_workers, pin_memory=pin_memory,
                           collate_fn=CollateFn(mode="re"))
    print("[DEBUG] RE train DataLoader created")
    sys.stdout.flush()

    # EL train DataLoader (if EL dataset is available)
    el_loader = None
    if el_train_ds is not None and len(el_train_ds) > 0:
        print("[DEBUG] Creating EL train DataLoader...")
        sys.stdout.flush()
        el_loader = DataLoader(el_train_ds, batch_size=args.batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=pin_memory,
                               collate_fn=CollateFn(mode="el"))
        print("[DEBUG] EL train DataLoader created")
        sys.stdout.flush()

    # Val loaders (shuffle=False) - only if val split is non-zero
    ner_val_loader = None
    re_val_loader = None
    el_val_loader = None
    if ner_val_ds is not None and len(ner_val_ds) > 0:
        ner_val_loader = DataLoader(ner_val_ds, batch_size=args.batch_size, shuffle=False,
                                    num_workers=num_workers, pin_memory=pin_memory,
                                    collate_fn=CollateFn(mode="ner"))
        print(f"[DEBUG] NER val DataLoader created ({len(ner_val_ds)} samples)")
    if re_val_ds is not None and len(re_val_ds) > 0:
        re_val_loader = DataLoader(re_val_ds, batch_size=args.batch_size, shuffle=False,
                                   num_workers=num_workers, pin_memory=pin_memory,
                                   collate_fn=CollateFn(mode="re"))
        print(f"[DEBUG] RE val DataLoader created ({len(re_val_ds)} samples)")
    if el_val_ds is not None and len(el_val_ds) > 0:
        el_val_loader = DataLoader(el_val_ds, batch_size=args.batch_size, shuffle=False,
                                   num_workers=num_workers, pin_memory=pin_memory,
                                   collate_fn=CollateFn(mode="el"))
        print(f"[DEBUG] EL val DataLoader created ({len(el_val_ds)} samples)")

    # Test loaders (shuffle=False) - only if test split is non-zero
    ner_test_loader = None
    re_test_loader = None
    el_test_loader = None
    if ner_test_ds is not None and len(ner_test_ds) > 0:
        ner_test_loader = DataLoader(ner_test_ds, batch_size=args.batch_size, shuffle=False,
                                     num_workers=num_workers, pin_memory=pin_memory,
                                     collate_fn=CollateFn(mode="ner"))
        print(f"[DEBUG] NER test DataLoader created ({len(ner_test_ds)} samples)")
    if re_test_ds is not None and len(re_test_ds) > 0:
        re_test_loader = DataLoader(re_test_ds, batch_size=args.batch_size, shuffle=False,
                                    num_workers=num_workers, pin_memory=pin_memory,
                                    collate_fn=CollateFn(mode="re"))
        print(f"[DEBUG] RE test DataLoader created ({len(re_test_ds)} samples)")
    if el_test_ds is not None and len(el_test_ds) > 0:
        el_test_loader = DataLoader(el_test_ds, batch_size=args.batch_size, shuffle=False,
                                    num_workers=num_workers, pin_memory=pin_memory,
                                    collate_fn=CollateFn(mode="el"))
        print(f"[DEBUG] EL test DataLoader created ({len(el_test_ds)} samples)")

    # Optimizer - projection heads + adapters + LoRA
    # Label embeddings are FROZEN to preserve pplx-embed semantics for zero-shot transfer
    head_lr = args.lr
    label_proj_lr = head_lr * 0.5  # Slower LR for label projections (near-identity init)
    adapter_lr = head_lr * 0.5  # Slower LR for adapters
    lora_lr = head_lr * 0.1  # Very slow LR for LoRA (small rank, sensitive)

    print(f"[DEBUG] Creating optimizer with head_lr={head_lr}, label_proj_lr={label_proj_lr}")
    print(f"[DEBUG] adapter_lr={adapter_lr}, lora_lr={lora_lr}")
    print(f"[DEBUG] Label embeddings are FROZEN (not in optimizer) for zero-shot transfer")
    sys.stdout.flush()

    optimizer_params = [
        {'params': model.ner_proj.parameters(), 'lr': head_lr, 'weight_decay': 0.01},
        {'params': model.ner_label_proj.parameters(), 'lr': label_proj_lr, 'weight_decay': 0.01},
        {'params': model.relation_proj.parameters(), 'lr': head_lr, 'weight_decay': 0.01},
        {'params': model.re_label_proj.parameters(), 'lr': label_proj_lr, 'weight_decay': 0.01},
        # Note: label embeddings are frozen, not included in optimizer
        {'params': model.ner_adapter.parameters(), 'lr': adapter_lr, 'weight_decay': 0.01},
        {'params': model.re_adapter.parameters(), 'lr': adapter_lr, 'weight_decay': 0.01},
        # EL components
        {'params': model.el_proj.parameters(), 'lr': head_lr, 'weight_decay': 0.01},
        {'params': model.el_label_proj.parameters(), 'lr': label_proj_lr, 'weight_decay': 0.01},
        # Note: EL label embeddings are frozen, not included in optimizer
        {'params': model.el_adapter.parameters(), 'lr': adapter_lr, 'weight_decay': 0.01},
        {'params': model.mention_gate.parameters(), 'lr': head_lr * 0.5, 'weight_decay': 0.01},
    ]

    # Add LoRA parameters if enabled
    if use_lora:
        for name, param in model.named_parameters():
            if 'lora' in name.lower() and param.requires_grad:
                optimizer_params.append({'params': param, 'lr': lora_lr, 'weight_decay': 0.01})

    optimizer = AdamW(optimizer_params)

    # Create scheduler with extended warmup for better stability
    # Calculate actual steps per epoch based on batch_size (use train splits)
    ner_len = len(ner_train_ds) // args.batch_size + (1 if len(ner_train_ds) % args.batch_size else 0)
    re_len = len(re_train_ds) // args.batch_size + (1 if len(re_train_ds) % args.batch_size else 0)
    steps_per_epoch = min(ner_len, re_len)
    num_training_steps = args.epochs * steps_per_epoch
    warmup_steps = max(1, int(num_training_steps * 0.1))  # 10% warmup (reduced for small step counts)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=num_training_steps
    )
    print(f"[SCHED] {steps_per_epoch} steps/epoch, {num_training_steps} total steps, {warmup_steps} warmup ({100*warmup_steps/num_training_steps:.0f}%)", flush=True)

    # === SPEED UP OPTIMIZATIONS ===
    # 1. AMP with BFloat16 - enabled immediately (no warmup delay)
    # BFloat16 has same dynamic range as FP32 but lower precision
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    # 2. Compile model for 10-30% speedup (PyTorch 2.0+)
    try:
        model = torch.compile(model, mode="reduce-overhead", fullgraph=False)
        compile_msg = "enabled (reduce-overhead mode)"
    except Exception as e:
        compile_msg = f"unavailable: {e}"

    print(f"[SPEED UP] AMP (bfloat16): {'enabled' if use_amp else 'disabled'} | torch.compile: {compile_msg}", flush=True)

    model.train()
    print(f"[TRAIN] Model in training mode. Starting {args.epochs} epochs...", flush=True)
    print(f"[LOSS] Using cosine similarity + cross-entropy loss with temperature annealing", flush=True)
    print(f"[LOSS] FROZEN label embeddings (preserve pplx-embed semantics for zero-shot)", flush=True)
    if zero_shot_eval:
        print(f"[LOSS] Zero-shot: ALL labels in softmax, frozen embeddings enable semantic transfer", flush=True)
    print(f"[LOSS] Direct label matrix lookup (no ANN during training)", flush=True)

    # Track best similarities for checkpoint saving
    best_ner_sim = -float('inf')
    best_re_sim = -float('inf')
    best_el_sim = -float('inf')
    best_combined_sim = -float('inf')
    best_ner_step = 0
    best_re_step = 0
    best_el_step = 0
    best_combined_step = 0

    # Running averages for smoother metrics
    ner_sim_window = []
    re_sim_window = []
    el_sim_window = []
    window_size = 20

    # Early stopping tracking
    best_loss = float('inf')
    patience_counter = 0

    for epoch in range(args.epochs):
        print(f"[TRAIN] Starting epoch {epoch}...", flush=True)

        # Temperature annealing: cosine decay for gentler schedule (prevents premature overconfidence)
        # Warmup steps = 10% of total training, then cosine decay
        warmup_epochs = max(1, int(args.epochs * 0.1))
        if epoch < warmup_epochs:
            # Constant high temp during warmup
            model.temperature = model.temperature_start
        else:
            # Cosine decay after warmup
            progress = (epoch - warmup_epochs) / (args.epochs - warmup_epochs)
            model.temperature = (model.temperature_end +
                                (model.temperature_start - model.temperature_end) *
                                0.5 * (1 + math.cos(math.pi * progress)))
        print(f"[TRAIN] Temperature: {model.temperature:.4f}", flush=True)

        # Compute lengths BEFORE creating iterators (len() with workers can hang)
        ner_len = len(ner_train_ds) // args.batch_size + (1 if len(ner_train_ds) % args.batch_size else 0)
        re_len = len(re_train_ds) // args.batch_size + (1 if len(re_train_ds) % args.batch_size else 0)
        el_len = len(el_train_ds) // args.batch_size + (1 if el_train_ds and len(el_train_ds) % args.batch_size else 0) if el_train_ds else 0
        steps = min(ner_len, re_len) if el_len == 0 else min(ner_len, re_len, el_len)
        print(f"[TRAIN] Epoch {epoch}: {steps} steps (NER batches: {ner_len}, RE batches: {re_len}, EL batches: {el_len})", flush=True)

        print(f"[TRAIN] Creating iterators (this may take time with multiprocessing)...", flush=True)
        ner_iter, re_iter = iter(ner_loader), iter(re_loader)
        el_iter = iter(el_loader) if el_loader else None
        print(f"[TRAIN] Iterators created, starting training loop...", flush=True)

        # Epoch statistics
        epoch_losses = []
        epoch_ner_sims = []
        epoch_re_sims = []
        epoch_el_sims = []

        for step in range(steps):
            global_step = epoch * steps + step

            if step == 0:
                print(f"[TRAIN] Starting first batch...", flush=True)
            n_batch = next(ner_iter)
            r_batch = next(re_iter)
            if n_batch is None or r_batch is None: continue

            if step == 0:
                print(f"[TRAIN] Got first batch, running forward pass...", flush=True)
                # Log VRAM usage
                if device.type == "cuda":
                    allocated = torch.cuda.memory_allocated(device) / 1e9
                    reserved = torch.cuda.memory_reserved(device) / 1e9
                    print(f"[VRAM] Allocated: {allocated:.2f}GB | Reserved: {reserved:.2f}GB", flush=True)

            optimizer.zero_grad()

            # Enable AMP with bfloat16 immediately (no warmup delay)
            autocast_enabled = use_amp
            if step == 0:
                print(f"[TRAIN] AMP {'enabled' if autocast_enabled else 'disabled'} for training", flush=True)
            with torch.amp.autocast("cuda", enabled=autocast_enabled, dtype=torch.bfloat16):
                # NER forward pass
                h_n = model.backbone(n_batch['input_ids'].to(device), n_batch['attention_mask'].to(device)).last_hidden_state
                if step == 0:
                    print(f"[TRAIN] NER backbone done, h_n shape={h_n.shape}", flush=True)
                n_preds = model.encode_spans_ner(h_n, n_batch['s_start'], n_batch['s_end'])
                if step == 0:
                    print(f"[TRAIN] NER preds done, n_preds shape={n_preds.shape}, has_nan={torch.isnan(n_preds).any()}", flush=True)

                # NER loss: Use zero-shot loss when heldout labels exist
                n_label_indices = n_batch['label_indices'].to(device)

                # === ZERO-SHOT: Use ALL labels in softmax for semantic transfer ===
                if zero_shot_eval and len(seen_ner_indices) < len(model.ner_label_embeddings):
                    # Zero-shot loss: ALL labels in softmax (no filtering)
                    # Frozen embeddings preserve pplx-embed semantics for heldout labels
                    n_loss, n_sim = cosine_cross_entropy_loss_zero_shot(
                        n_preds,
                        model.ner_label_embeddings,  # ALL labels (seen + heldout)
                        n_label_indices,
                        seen_indices=None,  # No filtering - all labels in softmax
                        temperature=model.temperature,
                        contrastive_weight=0.05
                    )
                else:
                    # Standard loss when no zero-shot testing
                    use_arcface = getattr(args, 'use_arcface', True)
                    if use_arcface:
                        n_loss, n_sim = arcface_loss(
                            n_preds,
                            model.ner_label_embeddings,
                            n_label_indices,
                            temperature=model.temperature,
                            margin_ner=0.05,
                            margin_re=0.0,
                            task_type="NER"
                        )
                    else:
                        n_loss, n_sim = cosine_cross_entropy_loss(
                            n_preds,
                            model.ner_label_embeddings,
                            n_label_indices,
                            temperature=model.temperature
                        )

                # RE forward pass
                h_r = model.backbone(r_batch['input_ids'].to(device), r_batch['attention_mask'].to(device)).last_hidden_state
                if step == 0:
                    print(f"[TRAIN] RE backbone done, h_r shape={h_r.shape}", flush=True)
                r_preds = model.encode_pairs_re(h_r, r_batch['s_start'], r_batch['s_end'], r_batch['o_start'], r_batch['o_end'])
                if step == 0:
                    print(f"[TRAIN] RE preds done, r_preds shape={r_preds.shape}, has_nan={torch.isnan(r_preds).any()}", flush=True)

                # RE loss: Use zero-shot loss when heldout labels exist
                r_label_indices = r_batch['label_indices'].to(device)

                # === ZERO-SHOT: Use ALL labels in softmax for semantic transfer ===
                if zero_shot_eval and len(seen_re_indices) < len(model.re_label_embeddings):
                    # Zero-shot loss: ALL labels in softmax (no filtering)
                    r_loss, r_sim = cosine_cross_entropy_loss_zero_shot(
                        r_preds,
                        model.re_label_embeddings,  # ALL labels (seen + heldout)
                        r_label_indices,
                        seen_indices=None,  # No filtering - all labels in softmax
                        temperature=model.temperature,
                        contrastive_weight=0.05
                    )
                else:
                    # Standard loss when no zero-shot testing
                    use_arcface = getattr(args, 'use_arcface', True)
                    if use_arcface:
                        r_loss, r_sim = arcface_loss(
                            r_preds,
                            model.re_label_embeddings,
                            r_label_indices,
                            temperature=model.temperature,
                            margin_ner=0.0,
                            margin_re=0.0,
                            task_type="RE"
                        )
                    else:
                        r_loss, r_sim = cosine_cross_entropy_loss(
                            r_preds,
                            model.re_label_embeddings,
                            r_label_indices,
                            temperature=model.temperature
                        )

                # EL forward pass (if EL data available)
                e_loss = torch.tensor(0.0, device=device)
                e_sim = 0.0
                if el_iter is not None:
                    try:
                        e_batch = next(el_iter)
                        if e_batch is not None:
                            h_e = model.backbone(e_batch['input_ids'].to(device), e_batch['attention_mask'].to(device)).last_hidden_state
                            if step == 0:
                                print(f"[TRAIN] EL backbone done, h_e shape={h_e.shape}", flush=True)
                            e_preds = model.encode_mentions_el(h_e, e_batch['s_start'], e_batch['s_end'])
                            if step == 0:
                                print(f"[TRAIN] EL preds done, e_preds shape={e_preds.shape}, has_nan={torch.isnan(e_preds).any()}", flush=True)

                            # EL loss: use chunked version for 6988 labels to avoid OOM
                            e_label_indices = e_batch['label_indices'].to(device)
                            # Only compute if we have EL labels
                            if model.el_label_embeddings.shape[0] > 0:
                                e_loss, e_sim = cosine_cross_entropy_loss_chunked(
                                    e_preds,
                                    model.el_label_embeddings,
                                    e_label_indices,
                                    temperature=model.temperature,
                                    chunk_size=1024  # Process 1024 labels at a time
                                )
                            else:
                                e_loss = torch.tensor(0.0, device=device)
                                e_sim = 0.0
                    except StopIteration:
                        # EL iterator exhausted, skip EL for remaining steps
                        pass

                if step == 0:
                    print(f"[TRAIN] Forward pass done. n_loss={n_loss:.4f}, r_loss={r_loss:.4f}, e_loss={e_loss:.4f}", flush=True)

                loss = n_loss + r_loss + e_loss

            # Check for NaN loss before backward
            if torch.isnan(loss):
                print(f"[TRAIN] NaN loss detected at epoch {epoch} step {step}, skipping", flush=True)
                continue

            # Backward pass (don't unscale yet - wait for NaN gradient check)
            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            # Check for NaN gradients BEFORE unscaling
            has_nan_grad = False
            for p in model.parameters():
                if p.grad is not None and torch.isnan(p.grad).any():
                    has_nan_grad = True
                    p.grad = None  # Clear the NaN gradient
                    break

            if has_nan_grad:
                print(f"[TRAIN] NaN gradient detected at epoch {epoch} step {step}, skipping step", flush=True)
                # Clear all gradients
                for p in model.parameters():
                    if p.grad is not None:
                        p.grad = None
                # IMPORTANT: Reset scaler state to prevent assertion error
                # When we skip after scale().backward(), we must call update() with new_scale
                if scaler is not None:
                    # Reduce scale factor and update scaler state properly
                    current_scale = scaler.get_scale()
                    scaler.update(new_scale=max(current_scale * 0.5, 1.0))
                continue

            # No NaN gradients - safe to proceed
            if scaler is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            # Re-normalize label embeddings (base + adapter) after each step
            # The adapter parameters are trainable, so we need to recompute combined embeddings
            with torch.no_grad():
                # Recombine base (frozen) + adapter (trainable) and normalize
                if model.ner_label_adapter is not None:
                    combined = F.normalize(
                        model.ner_label_embeddings_base + model.ner_label_adapter,
                        p=2, dim=-1, eps=1e-5
                    )
                    model.ner_label_embeddings = combined

                if model.re_label_adapter is not None:
                    combined = F.normalize(
                        model.re_label_embeddings_base + model.re_label_adapter,
                        p=2, dim=-1, eps=1e-5
                    )
                    model.re_label_embeddings = combined

                if model.el_label_adapter is not None:
                    combined = F.normalize(
                        model.el_label_embeddings_base + model.el_label_adapter,
                        p=2, dim=-1, eps=1e-5
                    )
                    model.el_label_embeddings = combined

            # Only update scheduler when we actually took a step
            scheduler.step()

            # Collect statistics
            epoch_losses.append(loss.item() if not torch.isnan(loss) else 0.0)
            epoch_ner_sims.append(n_sim)
            epoch_re_sims.append(r_sim)
            # Only track EL sim if EL was actually run (e_loss > 0)
            if isinstance(e_loss, torch.Tensor) and e_loss.item() > 0:
                epoch_el_sims.append(e_sim)
                el_sim_window.append(e_sim)
                if len(el_sim_window) > window_size:
                    el_sim_window.pop(0)

            # Update running averages
            ner_sim_window.append(n_sim)
            re_sim_window.append(r_sim)
            if len(ner_sim_window) > window_size:
                ner_sim_window.pop(0)
            if len(re_sim_window) > window_size:
                re_sim_window.pop(0)

            # Track best similarities
            if el_iter is not None and len(el_sim_window) > 0:
                combined_sim = (n_sim + r_sim + sum(el_sim_window) / len(el_sim_window)) / 3
            else:
                combined_sim = (n_sim + r_sim) / 2
            if n_sim > best_ner_sim:
                best_ner_sim = n_sim
                best_ner_step = global_step
            if r_sim > best_re_sim:
                best_re_sim = r_sim
                best_re_step = global_step
            if el_iter is not None and isinstance(e_loss, torch.Tensor) and e_loss.item() > 0:
                if e_sim > best_el_sim:
                    best_el_sim = e_sim
                    best_el_step = global_step
            if combined_sim > best_combined_sim:
                best_combined_sim = combined_sim
                best_combined_step = global_step

            # Print progress with running averages
            if step % 2 == 0:
                loss_val = loss.item() if not torch.isnan(loss) else 0.0
                ner_avg = sum(ner_sim_window) / len(ner_sim_window) if ner_sim_window else 0.0
                re_avg = sum(re_sim_window) / len(re_sim_window) if re_sim_window else 0.0
                el_avg = sum(el_sim_window) / len(el_sim_window) if el_sim_window else 0.0
                if el_iter is not None and len(el_sim_window) > 0:
                    print(f"Epoch {epoch} | Step {step:3d}/{steps} | Loss {loss_val:.4f} | Sim N:{n_sim:.3f}({ner_avg:.3f}) R:{r_sim:.3f}({re_avg:.3f}) E:{e_sim:.3f}({el_avg:.3f})", flush=True)
                else:
                    print(f"Epoch {epoch} | Step {step:3d}/{steps} | Loss {loss_val:.4f} | Sim N:{n_sim:.3f}({ner_avg:.3f}) R:{r_sim:.3f}({re_avg:.3f})", flush=True)

        # Epoch summary
        avg_loss = sum(epoch_losses) / len(epoch_losses)
        avg_ner_sim = sum(epoch_ner_sims) / len(epoch_ner_sims)
        avg_re_sim = sum(epoch_re_sims) / len(epoch_re_sims)
        max_ner_sim = max(epoch_ner_sims)
        max_re_sim = max(epoch_re_sims)
        avg_el_sim = sum(epoch_el_sims) / len(epoch_el_sims) if epoch_el_sims else 0.0
        max_el_sim = max(epoch_el_sims) if epoch_el_sims else 0.0
        if el_iter is not None and epoch_el_sims:
            print(f"[TRAIN] Epoch {epoch} completed - Avg Loss: {avg_loss:.4f} | Sim N: {avg_ner_sim:.3f} (max {max_ner_sim:.3f}) | Sim R: {avg_re_sim:.3f} (max {max_re_sim:.3f}) | Sim E: {avg_el_sim:.3f} (max {max_el_sim:.3f})", flush=True)
        else:
            print(f"[TRAIN] Epoch {epoch} completed - Avg Loss: {avg_loss:.4f} | Sim N: {avg_ner_sim:.3f} (max {max_ner_sim:.3f}) | Sim R: {avg_re_sim:.3f} (max {max_re_sim:.3f})", flush=True)

        # === VALIDATION EVALUATION ===
        if ner_val_loader is not None or re_val_loader is not None:
            val_metrics = evaluate_val_loss(
                model, ner_val_loader, re_val_loader, el_val_loader,
                device, args, zero_shot_eval, seen_ner_indices, seen_re_indices,
                max_batches=20
            )
            avg_val_loss = val_metrics['val_loss']
            print(f"[VAL] Epoch {epoch} | Val Loss: {avg_val_loss:.4f} | Val Sim N: {val_metrics['val_ner_sim']:.3f} | Val Sim R: {val_metrics['val_re_sim']:.3f}", flush=True)
            if el_val_loader is not None:
                print(f"[VAL] Epoch {epoch} | Val Sim E: {val_metrics['val_el_sim']:.3f}", flush=True)
            gap = avg_val_loss - avg_loss
            print(f"[COMPARE] Epoch {epoch} | Train Loss: {avg_loss:.4f} vs Val Loss: {avg_val_loss:.4f} | Gap: {gap:+.4f}", flush=True)
        else:
            avg_val_loss = avg_loss

        # === ZERO-SHOT EVALUATION ===
        if zero_shot_eval and (heldout_ner_labels or heldout_re_labels or heldout_el_labels):
            print(f"[ZERO-SHOT] Evaluating zero-shot transfer to heldout labels...", flush=True)

            # Evaluate NER zero-shot (use val loader if available, else train loader)
            _ner_eval_loader = ner_val_loader if ner_val_loader is not None else ner_loader
            if heldout_ner_labels:
                ner_metrics = evaluate_zero_shot(
                    model, _ner_eval_loader, device,
                    seen_ner_labels, heldout_ner_labels,
                    mode="NER", temperature=model.temperature, max_batches=5
                )
                print(f"[ZERO-SHOT] Epoch {epoch} | NER Seen: {ner_metrics['seen_sim']:.3f} | "
                      f"Heldout: {ner_metrics['heldout_sim']:.3f} | Gap: {ner_metrics['zero_shot_gap']:.3f}", flush=True)

            # Evaluate RE zero-shot (use val loader if available, else train loader)
            _re_eval_loader = re_val_loader if re_val_loader is not None else re_loader
            if heldout_re_labels:
                re_metrics = evaluate_zero_shot(
                    model, _re_eval_loader, device,
                    seen_re_labels, heldout_re_labels,
                    mode="RE", temperature=model.temperature, max_batches=5
                )
                print(f"[ZERO-SHOT] Epoch {epoch} | RE Seen: {re_metrics['seen_sim']:.3f} | "
                      f"Heldout: {re_metrics['heldout_sim']:.3f} | Gap: {re_metrics['zero_shot_gap']:.3f}", flush=True)

            # Evaluate EL zero-shot (use val loader if available, else train loader)
            _el_eval_loader = el_val_loader if el_val_loader is not None else el_loader
            if heldout_el_labels and _el_eval_loader is not None:
                el_metrics = evaluate_zero_shot(
                    model, _el_eval_loader, device,
                    seen_el_labels, heldout_el_labels,
                    mode="EL", temperature=model.temperature, max_batches=5
                )
                print(f"[ZERO-SHOT] Epoch {epoch} | EL Seen: {el_metrics['seen_sim']:.3f} | "
                      f"Heldout: {el_metrics['heldout_sim']:.3f} | Gap: {el_metrics['zero_shot_gap']:.3f}", flush=True)

        # Save checkpoint after each epoch
        checkpoint_path = os.path.join(args.output_dir, f"checkpoint_epoch_{epoch}.pt")
        checkpoint_data = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'args': args,
            'ner_label2idx': model.ner_label2idx,
            're_label2idx': model.re_label2idx,
            'el_label2idx': model.el_label2idx,
            'temperature': model.temperature,
            'train_loss': avg_loss,
            'val_loss': avg_val_loss,
        }
        # Save zero-shot heldout label info for analysis
        if zero_shot_eval:
            checkpoint_data.update({
                'seen_ner_labels': list(seen_ner_labels),
                'heldout_ner_labels': list(heldout_ner_labels),
                'seen_re_labels': list(seen_re_labels),
                'heldout_re_labels': list(heldout_re_labels),
                'seen_el_labels': list(seen_el_labels),
                'heldout_el_labels': list(heldout_el_labels),
            })
        torch.save(checkpoint_data, checkpoint_path)
        print(f"[SAVE] Checkpoint saved to {checkpoint_path}", flush=True)

        # Save best model based on combined similarity
        if el_iter is not None and epoch_el_sims:
            combined_sims = avg_ner_sim + avg_re_sim + avg_el_sim
            if combined_sims > best_combined_sim * 3:
                best_path = os.path.join(args.output_dir, "model_best.pt")
                torch.save({
                    'epoch': epoch,
                    'global_step': global_step,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_ner_sim': best_ner_sim,
                    'best_re_sim': best_re_sim,
                    'best_el_sim': best_el_sim,
                    'ner_sim': avg_ner_sim,
                    're_sim': avg_re_sim,
                    'el_sim': avg_el_sim,
                    'args': args,
                    'ner_label2idx': model.ner_label2idx,
                    're_label2idx': model.re_label2idx,
                    'el_label2idx': model.el_label2idx,
                    'temperature': model.temperature,
                }, best_path)
                print(f"[BEST] New best model saved! (N: {avg_ner_sim:.3f}, R: {avg_re_sim:.3f}, E: {avg_el_sim:.3f})", flush=True)
        else:
            # Original behavior without EL
            if avg_ner_sim + avg_re_sim > best_combined_sim * 2:
                best_path = os.path.join(args.output_dir, "model_best.pt")
                torch.save({
                    'epoch': epoch,
                    'global_step': global_step,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_ner_sim': best_ner_sim,
                    'best_re_sim': best_re_sim,
                    'ner_sim': avg_ner_sim,
                    're_sim': avg_re_sim,
                    'args': args,
                    'ner_label2idx': model.ner_label2idx,
                    're_label2idx': model.re_label2idx,
                    'el_label2idx': model.el_label2idx,
                    'temperature': model.temperature,
                }, best_path)
                print(f"[BEST] New best model saved! (N: {avg_ner_sim:.3f}, R: {avg_re_sim:.3f})", flush=True)

        # === EARLY STOPPING ===
        # Stop training if val loss (or train loss if no val split) hasn't improved for patience epochs
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            patience_counter = 0
            print(f"[EARLY STOP] New best loss: {best_loss:.4f}", flush=True)
        else:
            patience_counter += 1
            print(f"[EARLY STOP] No improvement for {patience_counter}/{args.early_stop_patience} epochs", flush=True)
            if patience_counter >= args.early_stop_patience:
                print(f"[EARLY STOP] Training stopped early - no improvement for {patience_counter} epochs", flush=True)
                break

    # === FINAL TEST EVALUATION ===
    if ner_test_loader is not None or re_test_loader is not None:
        print(f"\n{'='*60}", flush=True)
        print(f"[TEST] Final evaluation on held-out test set...", flush=True)
        test_metrics = evaluate_val_loss(
            model, ner_test_loader, re_test_loader, el_test_loader,
            device, args, zero_shot_eval, seen_ner_indices, seen_re_indices,
            max_batches=50
        )
        print(f"[TEST] Test Loss: {test_metrics['val_loss']:.4f}", flush=True)
        print(f"[TEST] Test Sim N: {test_metrics['val_ner_sim']:.3f} | Test Sim R: {test_metrics['val_re_sim']:.3f}", flush=True)
        if el_test_loader is not None:
            print(f"[TEST] Test Sim E: {test_metrics['val_el_sim']:.3f}", flush=True)

        # Zero-shot evaluation on test set
        if zero_shot_eval:
            if heldout_ner_labels and ner_test_loader is not None:
                ner_test_metrics = evaluate_zero_shot(
                    model, ner_test_loader, device,
                    seen_ner_labels, heldout_ner_labels,
                    mode="NER", temperature=model.temperature, max_batches=10
                )
                print(f"[TEST-ZS] NER Seen: {ner_test_metrics['seen_sim']:.3f} | "
                      f"Heldout: {ner_test_metrics['heldout_sim']:.3f} | Gap: {ner_test_metrics['zero_shot_gap']:.3f}", flush=True)
            if heldout_re_labels and re_test_loader is not None:
                re_test_metrics = evaluate_zero_shot(
                    model, re_test_loader, device,
                    seen_re_labels, heldout_re_labels,
                    mode="RE", temperature=model.temperature, max_batches=10
                )
                print(f"[TEST-ZS] RE Seen: {re_test_metrics['seen_sim']:.3f} | "
                      f"Heldout: {re_test_metrics['heldout_sim']:.3f} | Gap: {re_test_metrics['zero_shot_gap']:.3f}", flush=True)
            if heldout_el_labels and el_test_loader is not None:
                el_test_metrics = evaluate_zero_shot(
                    model, el_test_loader, device,
                    seen_el_labels, heldout_el_labels,
                    mode="EL", temperature=model.temperature, max_batches=10
                )
                print(f"[TEST-ZS] EL Seen: {el_test_metrics['seen_sim']:.3f} | "
                      f"Heldout: {el_test_metrics['heldout_sim']:.3f} | Gap: {el_test_metrics['zero_shot_gap']:.3f}", flush=True)
        print(f"{'='*60}", flush=True)

    # Save final model
    final_path = os.path.join(args.output_dir, "model_final.pt")
    torch.save({
        'model_state_dict': model.state_dict(),
        'args': args,
        'ner_label2idx': model.ner_label2idx,
        're_label2idx': model.re_label2idx,
        'el_label2idx': model.el_label2idx,
        'temperature': model.temperature,
    }, final_path)
    print(f"[SAVE] Final model saved to {final_path}", flush=True)

    # Also save just the projection heads for easy loading
    heads_path = os.path.join(args.output_dir, "projection_heads.pt")
    heads_data = {
        'ner_proj': model.ner_proj.state_dict(),
        'ner_label_proj': model.ner_label_proj.state_dict(),  # Label projection
        'ner_adapter': model.ner_adapter.state_dict(),  # NER task adapter
        'relation_proj': model.relation_proj.state_dict(),  # RE relation projection
        're_label_proj': model.re_label_proj.state_dict(),  # Label projection
        're_adapter': model.re_adapter.state_dict(),  # RE task adapter
        'el_proj': model.el_proj.state_dict(),  # EL projection
        'el_label_proj': model.el_label_proj.state_dict(),  # EL label projection
        'el_adapter': model.el_adapter.state_dict(),  # EL task adapter
        'mention_gate': model.mention_gate.state_dict(),  # Mention gate
        'ner_label_embeddings': model.ner_label_embeddings,  # Frozen labels
        're_label_embeddings': model.re_label_embeddings,     # Frozen labels
        'el_label_embeddings': model.el_label_embeddings,     # EL labels
        'ner_label2idx': model.ner_label2idx,
        're_label2idx': model.re_label2idx,
        'el_label2idx': model.el_label2idx,
        'temperature': model.temperature,
        'args': args,
    }
    # Save zero-shot heldout label info for analysis
    if zero_shot_eval:
        heads_data.update({
            'seen_ner_labels': list(seen_ner_labels),
            'heldout_ner_labels': list(heldout_ner_labels),
            'seen_re_labels': list(seen_re_labels),
            'heldout_re_labels': list(heldout_re_labels),
            'seen_el_labels': list(seen_el_labels),
            'heldout_el_labels': list(heldout_el_labels),
        })
    torch.save(heads_data, heads_path)
    print(f"[SAVE] Projection heads saved to {heads_path}", flush=True)

    # Print training summary
    print(f"\n{'='*60}", flush=True)
    print(f"[SUMMARY] Training complete!", flush=True)
    print(f"  Best NER similarity: {best_ner_sim:.4f} at step {best_ner_step}", flush=True)
    print(f"  Best RE similarity:  {best_re_sim:.4f} at step {best_re_step}", flush=True)
    if best_el_sim > -float('inf'):
        print(f"  Best EL similarity:  {best_el_sim:.4f} at step {best_el_step}", flush=True)
    print(f"  Best combined sim:   {best_combined_sim:.4f} at step {best_combined_step}", flush=True)

    # Zero-shot summary
    if zero_shot_eval:
        print(f"\n[ZERO-SHOT] Heldout label summary:", flush=True)
        if heldout_ner_labels:
            print(f"  NER heldout: {heldout_ner_labels} ({len(heldout_ner_labels)} labels)", flush=True)
        if heldout_re_labels:
            print(f"  RE heldout: {heldout_re_labels} ({len(heldout_re_labels)} labels)", flush=True)
        if heldout_el_labels:
            print(f"  EL heldout: {len(heldout_el_labels)} entities (zero-shot KB entities)", flush=True)
        print(f"  Checkpoint includes seen/heldout label info for analysis", flush=True)

    # VRAM usage info for batch size tuning
    if device.type == "cuda":
        peak_allocated = torch.cuda.max_memory_allocated(device) / 1e9
        peak_reserved = torch.cuda.max_memory_reserved(device) / 1e9
        total_vram = torch.cuda.get_device_properties(device).total_memory / 1e9
        util_pct = 100 * peak_reserved / total_vram
        print(f"  Peak VRAM: {peak_reserved:.2f}GB / {total_vram:.2f}GB ({util_pct:.0f}%)", flush=True)
        if util_pct < 70:
            print(f"  [TIP] You have spare VRAM! Try --batch_size {args.batch_size * 2} for faster training", flush=True)

    print(f"{'='*60}\n", flush=True)
    print(f"[DONE] All models saved to {args.output_dir}/", flush=True)


def strip_compile_prefix(state_dict):
    """Strip '_orig_mod.' prefix from state_dict keys added by torch.compile().

    Also handles LoRA keys where '_orig_mod' appears nested within
LoRA layer paths.
    """
    new_state_dict = {}
    for k, v in state_dict.items():
        # Strip _orig_mod. prefix recursively from key path
        parts = k.split(".")
        new_parts = []
        for part in parts:
            if part.startswith("_orig_mod"):
                new_parts.append(part[5:])  # strip _orig_mod prefix
            else:
                new_parts.append(part)
        new_key = ".".join(new_parts)
        new_state_dict[new_key] = v
    return new_state_dict


def load_trained_model(checkpoint_path, model_name="perplexity-ai/pplx-embed-context-v1-0.6b",
                      tokenizer=None, device=None):
    """Load a trained model from checkpoint for inference."""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load checkpoint first to get label mappings
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    ner_label2idx = checkpoint.get('ner_label2idx', {})
    re_label2idx = checkpoint.get('re_label2idx', {})
    el_label2idx = checkpoint.get('el_label2idx', {})

    # Create model with label mappings
    model = UnifiedKGModel(
        model_name=model_name,
        ner_label2idx=ner_label2idx,
        re_label2idx=re_label2idx,
        el_label2idx=el_label2idx,
        tokenizer=tokenizer,
        device=device
    ).to(device)

    # Initialize frozen labels if tokenizer provided
    if tokenizer is not None:
        model.initialize_frozen_labels(tokenizer, device)

    if 'model_state_dict' in checkpoint:
        # Full model checkpoint - strip compile prefix and LoRA/compile artifacts
        clean_sd = strip_compile_prefix(checkpoint['model_state_dict'])
        model.load_state_dict(clean_sd, strict=False)
    else:
        # Projection heads only checkpoint
        model.ner_proj.load_state_dict(checkpoint['ner_proj'])
        if 'relation_proj' in checkpoint:
            model.relation_proj.load_state_dict(checkpoint['relation_proj'])
        elif 'entity_span_pooler' in checkpoint:
            # Legacy checkpoint compatibility - skip removed modules
            pass
        # Load label projections if available (new format)
        if 'ner_label_proj' in checkpoint:
            model.ner_label_proj.load_state_dict(checkpoint['ner_label_proj'])
        if 're_label_proj' in checkpoint:
            model.re_label_proj.load_state_dict(checkpoint['re_label_proj'])
        # Load EL projections if available
        if 'el_proj' in checkpoint:
            model.el_proj.load_state_dict(checkpoint['el_proj'])
        if 'el_label_proj' in checkpoint:
            model.el_label_proj.load_state_dict(checkpoint['el_label_proj'])
        # Load adapters if available
        if 'ner_adapter' in checkpoint:
            model.ner_adapter.load_state_dict(checkpoint['ner_adapter'])
        if 're_adapter' in checkpoint:
            model.re_adapter.load_state_dict(checkpoint['re_adapter'])
        if 'el_adapter' in checkpoint:
            model.el_adapter.load_state_dict(checkpoint['el_adapter'])
        if 'mention_gate' in checkpoint:
            model.mention_gate.load_state_dict(checkpoint['mention_gate'])
        # Load frozen labels
        if 'ner_label_embeddings' in checkpoint:
            model.ner_label_embeddings = checkpoint['ner_label_embeddings']
        if 're_label_embeddings' in checkpoint:
            model.re_label_embeddings = checkpoint['re_label_embeddings']
        if 'el_label_embeddings' in checkpoint:
            model.el_label_embeddings = checkpoint['el_label_embeddings']

    model.eval()
    print(f"[LOAD] Model loaded from {checkpoint_path}")
    return model


def extract_entities(text, model, tokenizer, entity_types, device=None, top_k=5):
    """Extract entities from text using trained model."""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Encode entity type labels with task prefix
    prefixed_types = [f"{NER_PREFIX}{t}" for t in entity_types]
    with torch.no_grad():
        label_vecs = model.encode_frozen_labels(prefixed_types, tokenizer, device)

    # Tokenize text
    enc = tokenizer(text, return_offsets_mapping=True, max_length=512, truncation=True)
    input_ids = torch.tensor([enc['input_ids']]).to(device)
    attention_mask = torch.tensor([enc['attention_mask']]).to(device)

    # Get token embeddings
    with torch.no_grad():
        hidden = model.backbone(input_ids, attention_mask).last_hidden_state

        # Try all possible spans (limit to reasonable length)
        results = []
        max_span_len = 20
        for start in range(1, min(len(enc['offset_mapping']), max_span_len)):
            for end in range(start, min(start + max_span_len, len(enc['offset_mapping']))):
                span_vec = model.encode_spans_ner(hidden, torch.tensor([start]), torch.tensor([end]))

                # Compare to all entity types
                sims = (span_vec @ label_vecs.T).squeeze(0)
                best_score = sims.max().item()

                if best_score > 0.3:  # Threshold
                    best_type_idx = sims.argmax().item()
                    char_start, char_end = enc['offset_mapping'][start][0], enc['offset_mapping'][end][1]
                    entity_text = text[char_start:char_end]

                    results.append({
                        'text': entity_text,
                        'type': entity_types[best_type_idx],
                        'score': best_score,
                        'span': (char_start, char_end)
                    })

    # Deduplicate and sort by score
    seen = set()
    unique_results = []
    for r in sorted(results, key=lambda x: -x['score']):
        key = (r['text'], r['type'])
        if key not in seen:
            seen.add(key)
            unique_results.append(r)
            if len(unique_results) >= top_k:
                break

    return unique_results

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", default="perplexity-ai/pplx-embed-context-v1-0.6b")
    parser.add_argument("--sample_size", type=int, default=50000)  # More data for better learning
    parser.add_argument("--batch_size", type=int, default=768)  # Increased from 512 - VRAM headroom available
    parser.add_argument("--epochs", type=int, default=5)  # More epochs for convergence
    parser.add_argument("--early_stop_patience", type=int, default=8)  # More patience for slower convergence
    parser.add_argument("--lr", type=float, default=1e-4)  # Higher LR for faster convergence
    parser.add_argument("--max_length", type=int, default=128)
    parser.add_argument("--output_dir", type=str, default="./checkpoints")

    # New arguments for improved training
    parser.add_argument("--use_lora", type=lambda x: x.lower() == 'true', default=False,
                        help="Enable LoRA adapters on backbone attention layers")
    parser.add_argument("--lora_rank", type=int, default=8,
                        help="LoRA adapter rank (default: 8)")
    parser.add_argument("--lora_alpha", type=int, default=16,
                        help="LoRA adapter alpha (default: 16)")
    parser.add_argument("--use_arcface", type=lambda x: x.lower() == 'true', default=True,
                        help="Use ArcFace loss with adaptive margins")

    # Zero-shot heldout label testing arguments
    parser.add_argument("--heldout_ner_labels", type=str, default="LOC",
                        help="Comma-separated NER labels to hold out (default: LOC)")
    parser.add_argument("--heldout_re_labels", type=str, default="P150,located_in,founded_by",
                        help="Comma-separated RE labels to hold out (default: P150,located_in,founded_by)")
    parser.add_argument("--heldout_el_labels", type=str, default="",
                        help="Comma-separated EL entity labels to hold out (default: random 15%% sampling)")
    parser.add_argument("--heldout_el_ratio", type=float, default=DEFAULT_HELDOUT_EL_RATIO,
                        help=f"Fraction of EL entities to hold out (default: {DEFAULT_HELDOUT_EL_RATIO})")
    parser.add_argument("--heldout_ratio", type=float, default=0.15,
                        help="Fraction of labels to hold out if defaults not found (default: 0.15)")
    parser.add_argument("--zero_shot_eval", action="store_true", default=True,
                        help="Enable zero-shot evaluation during training (default: True)")
    parser.add_argument("--disable_zero_shot", action="store_true",
                        help="Disable zero-shot testing - train on all labels")
    parser.add_argument("--use_el", type=lambda x: x.lower() == 'true', default=True,
                        help="Enable Entity Linking task (default: True)")
    parser.add_argument("--el_sample_size", type=int, default=20000,
                        help="EL training samples per dataset (default: 20000)")

    # Train/val/test split arguments
    parser.add_argument("--val_split", type=float, default=0.1,
                        help="Fraction of data for validation (default: 0.1)")
    parser.add_argument("--test_split", type=float, default=0.1,
                        help="Fraction of data for final test evaluation (default: 0.1)")
    parser.add_argument("--split_seed", type=int, default=42,
                        help="Random seed for reproducible data splits (default: 42)")

    # === GENERATIVE DECODER TRAINING ARGUMENTS ===
    parser.add_argument("--train_decoder", default=True, action="store_true",
                        help="Train decoder instead of main model (requires pre-trained main model)")
    parser.add_argument("--decoder_data", type=str, default="./cache/decoder_data.pt",
                        help="Path to pre-computed decoder training data")
    parser.add_argument("--decoder_epochs", type=int, default=5,
                        help="Number of decoder training epochs (default: 5)")
    parser.add_argument("--decoder_batch_size", type=int, default=32,
                        help="Batch size for decoder training (default: 32)")
    parser.add_argument("--decoder_lr", type=float, default=1e-4,
                        help="Learning rate for decoder (default: 1e-4)")
    parser.add_argument("--decoder_max_len", type=int, default=16,
                        help="Max length for generated labels (default: 16)")
    parser.add_argument("--lambda_emb", type=float, default=1.0,
                        help="Weight for embedding loss (default: 1.0)")
    parser.add_argument("--lambda_text", type=float, default=1.0,
                        help="Weight for text generation loss (default: 1.0)")
    parser.add_argument("--prepare_decoder_data", default=False, action="store_true",
                        help="Prepare decoder training data from trained model")
    parser.add_argument("--decoder_checkpoint", default="./checkpoints/model_best.pt", type=str, #default=None,
                        help="Path to main model checkpoint for decoder training")

    args, _ = parser.parse_known_args()

    # Create output directory
    os.makedirs(args.output_dir, exist_ok=True)

    # Print configuration
    print(f"\n{'='*60}")
    print(f"[CONFIG] Training Configuration:")
    print(f"  Model: {args.model_name}")
    print(f"  Sample size: {args.sample_size}")
    print(f"  Batch size: {args.batch_size}")
    print(f"  Epochs: {args.epochs}")
    print(f"  Learning rate: {args.lr}")
    print(f"  LoRA: {'enabled' if args.use_lora else 'disabled'} (rank={args.lora_rank}, alpha={args.lora_alpha})")
    print(f"  ArcFace loss: {'enabled' if args.use_arcface else 'disabled'}")
    print(f"  Zero-shot testing: {'disabled' if args.disable_zero_shot else 'enabled'}")
    if not args.disable_zero_shot:
        print(f"    Heldout NER labels: {args.heldout_ner_labels}")
        print(f"    Heldout RE labels: {args.heldout_re_labels}")
        print(f"    Heldout EL ratio: {args.heldout_el_ratio}")
    print(f"  Entity Linking: {'enabled' if args.use_el else 'disabled'} (sample_size={args.el_sample_size})")
    print(f"  Data splits: val={args.val_split}, test={args.test_split}, seed={args.split_seed}")
    # Decoder training configuration
    if args.train_decoder or args.prepare_decoder_data:
        print(f"  Decoder training: {'enabled' if args.train_decoder else 'disabled'}")
        print(f"  Decoder data prep: {'enabled' if args.prepare_decoder_data else 'disabled'}")
        if args.train_decoder:
            print(f"    Decoder LR: {args.decoder_lr}, Epochs: {args.decoder_epochs}")
            print(f"    Lambda emb: {args.lambda_emb}, Lambda text: {args.lambda_text}")
    print(f"{'='*60}\n")

    # Handle decoder training modes
    if args.train_decoder:
        # Load pre-trained main model and train decoder
        if args.decoder_checkpoint is None:
            print("[ERROR] --decoder_checkpoint required when using --train_decoder")
            print("[ERROR] Please provide path to trained main model checkpoint")
            sys.exit(1)

        print(f"[DECODER] Loading main model from {args.decoder_checkpoint}")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        # Load model with decoder
        checkpoint = torch.load(args.decoder_checkpoint, map_location=device, weights_only=False)
        ner_label2idx = checkpoint.get('ner_label2idx', {})
        re_label2idx = checkpoint.get('re_label2idx', {})
        el_label2idx = checkpoint.get('el_label2idx', {})

        model = UnifiedKGModel(
            model_name=args.model_name,
            ner_label2idx=ner_label2idx,
            re_label2idx=re_label2idx,
            el_label2idx=el_label2idx,
            tokenizer=tokenizer,
            device=device
        ).to(device)

        # Initialize frozen labels
        model.initialize_frozen_labels(tokenizer, device)

        # Load model weights
        if 'model_state_dict' in checkpoint:
            clean_sd = strip_compile_prefix(checkpoint['model_state_dict'])
            model.load_state_dict(clean_sd, strict=False)
        else:
            # Load projection heads
            model.ner_proj.load_state_dict(checkpoint['ner_proj'])
            if 'relation_proj' in checkpoint:
                model.relation_proj.load_state_dict(checkpoint['relation_proj'])
            if 'ner_label_proj' in checkpoint:
                model.ner_label_proj.load_state_dict(checkpoint['ner_label_proj'])
            if 're_label_proj' in checkpoint:
                model.re_label_proj.load_state_dict(checkpoint['re_label_proj'])
            if 'el_proj' in checkpoint:
                model.el_proj.load_state_dict(checkpoint['el_proj'])
            if 'el_label_proj' in checkpoint:
                model.el_label_proj.load_state_dict(checkpoint['el_label_proj'])

        # Initialize decoder
        model.initialize_decoder(tokenizer, device)

        # Check if decoder data exists
        if not os.path.exists(args.decoder_data):
            print(f"[ERROR] Decoder training data not found at {args.decoder_data}")
            print(f"[ERROR] Please run with --prepare_decoder_data first")
            sys.exit(1)

        # Train decoder
        print(f"[DECODER] Starting decoder training...")
        metrics = train_decoder(model, args.decoder_data, tokenizer, device, args)

        # Save model with trained decoder
        output_path = os.path.join(args.output_dir, "model_with_decoder.pt")
        torch.save({
            'model_state_dict': model.state_dict(),
            'decoder_metrics': metrics,
            'args': args,
        }, output_path)
        print(f"[SAVE] Model with trained decoder saved to {output_path}")

        print(f"\n{'='*60}")
        print(f"[SUMMARY] Decoder training complete!")
        print(f"  Final loss: {metrics['final_loss']:.4f}")
        print(f"  Embedding loss: {metrics['final_emb_loss']:.4f}")
        print(f"  Text loss: {metrics['final_text_loss']:.4f}")
        print(f"{'='*60}\n")

    elif args.prepare_decoder_data:
        # Prepare decoder training data from existing model
        if args.decoder_checkpoint is None:
            print("[ERROR] --decoder_checkpoint required when using --prepare_decoder_data")
            print("[ERROR] Please provide path to trained main model checkpoint")
            sys.exit(1)

        print(f"[DECODER PREP] Loading main model from {args.decoder_checkpoint}")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        # Load model
        checkpoint = torch.load(args.decoder_checkpoint, map_location=device, weights_only=False)
        ner_label2idx = checkpoint.get('ner_label2idx', {})
        re_label2idx = checkpoint.get('re_label2idx', {})
        el_label2idx = checkpoint.get('el_label2idx', {})

        model = UnifiedKGModel(
            model_name=args.model_name,
            ner_label2idx=ner_label2idx,
            re_label2idx=re_label2idx,
            el_label2idx=el_label2idx,
            tokenizer=tokenizer,
            device=device
        ).to(device)

        model.initialize_frozen_labels(tokenizer, device)

        if 'model_state_dict' in checkpoint:
            clean_sd = strip_compile_prefix(checkpoint['model_state_dict'])
            model.load_state_dict(clean_sd, strict=False)
        else:
            model.ner_proj.load_state_dict(checkpoint['ner_proj'])
            if 'relation_proj' in checkpoint:
                model.relation_proj.load_state_dict(checkpoint['relation_proj'])
            if 'ner_label_proj' in checkpoint:
                model.ner_label_proj.load_state_dict(checkpoint['ner_label_proj'])
            if 're_label_proj' in checkpoint:
                model.re_label_proj.load_state_dict(checkpoint['re_label_proj'])
            if 'el_proj' in checkpoint:
                model.el_proj.load_state_dict(checkpoint['el_proj'])
            if 'el_label_proj' in checkpoint:
                model.el_label_proj.load_state_dict(checkpoint['el_label_proj'])

        # Load datasets for preparing decoder data
        print("[DECODER PREP] Loading datasets...")
        from datasets import load_dataset

        # Load small subset of data for decoder preparation
        ner_raw = []
        for item in load_dataset("conll2003", split="train", streaming=True, revision="refs/convert/parquet").take(1000):
            # Convert CoNLL-2003 token/tag format to text+entities format
            tokens = item['tokens']
            ner_tags = item['ner_tags']
            text = " ".join(tokens)
            tag_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
            entities = []
            i = 0
            while i < len(tokens):
                tag_val = ner_tags[i]
                if 0 <= tag_val < len(tag_names):
                    tag_str = tag_names[tag_val]
                else:
                    tag_str = 'O'
                if tag_str.startswith('B-'):
                    ent_type = tag_str[2:]
                    ent_tokens = [tokens[i]]
                    j = i + 1
                    while j < len(tokens):
                        tv = ner_tags[j]
                        ts = tag_names[tv] if 0 <= tv < len(tag_names) else 'O'
                        if ts == f'I-{ent_type}':
                            ent_tokens.append(tokens[j])
                            j += 1
                        else:
                            break
                    entities.append({'entity': " ".join(ent_tokens), 'type': ent_type})
                    i = j
                else:
                    i += 1
            if entities:
                ner_raw.append({'text': text, 'entities': entities})
        re_raw = []
        for item in load_dataset("tonytan48/Re-DocRED", split="train", streaming=True).take(1000):
            re_raw.append(item)

        # Create datasets
        ner_ds = UnifiedDataset(ner_raw, tokenizer, "NER", max_length=args.max_length, sample_size=1000)
        re_ds = UnifiedDataset(re_raw, tokenizer, "RE", max_length=args.max_length, sample_size=1000)

        # Create dataloaders
        ner_loader = DataLoader(ner_ds, batch_size=args.batch_size, shuffle=False,
                                num_workers=0, collate_fn=CollateFn(mode="ner"))
        re_loader = DataLoader(re_ds, batch_size=args.batch_size, shuffle=False,
                               num_workers=0, collate_fn=CollateFn(mode="re"))

        dataloaders = {'ner': ner_loader, 're': re_loader}
        datasets = {'ner': ner_ds, 're': re_ds}

        # Prepare decoder data
        prepare_decoder_training_data(model, dataloaders, datasets, tokenizer, device,
                                     args.decoder_data, max_samples=5000)

        print(f"[DECODER PREP] Complete! Data saved to {args.decoder_data}")
        print(f"[DECODER PREP] Now run with --train_decoder to train the decoder")

    else:
        # Normal main model training
        train(args)


[CONFIG] Training Configuration:
  Model: perplexity-ai/pplx-embed-context-v1-0.6b
  Sample size: 50000
  Batch size: 768
  Epochs: 5
  Learning rate: 0.0001
  LoRA: disabled (rank=8, alpha=16)
  ArcFace loss: enabled
  Zero-shot testing: enabled
    Heldout NER labels: LOC
    Heldout RE labels: P150,located_in,founded_by
    Heldout EL ratio: 0.15
  Entity Linking: enabled (sample_size=20000)
  Data splits: val=0.1, test=0.1, seed=42
  Decoder training: enabled
  Decoder data prep: disabled
    Decoder LR: 0.0001, Epochs: 5
    Lambda emb: 1.0, Lambda text: 1.0

[DECODER] Loading main model from ./checkpoints/model_best.pt


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


The repository perplexity-ai/pplx-embed-context-v1-0.6b contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/perplexity-ai/pplx-embed-context-v1-0.6b .
 You can inspect the repository content at https://hf.co/perplexity-ai/pplx-embed-context-v1-0.6b.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[LORA] Applied LoRA to 84 backbone layers (rank=8, alpha=16)
[INIT] Encoding 4 NER labels with pplx-embed...
[INIT] NER labels: ['[NER] LOC', '[NER] MISC', '[NER] ORG', '[NER] PER']
[INIT] Encoding 94 REL labels with pplx-embed...
[INIT] REL labels: ['[REL] P1001', '[REL] P102', '[REL] P1056', '[REL] P108', '[REL] P112', '[REL] P118', '[REL] P123', '[REL] P127', '[REL] P131', '[REL] P1336']...
[INIT] Encoding 9994 EL labels with pplx-embed...
[INIT] EL labels: ["[LINK] # 1 's", '[LINK] $ 1000', '[LINK] ( February 12 , 1817', "[LINK] ( We 've Got ) Honey Love", '[LINK] -1937', '[LINK] .Mac', '[LINK] /18 May 1888', '[LINK] 1 April 1748', '[LINK] 1 April 1961', '[LINK] 1 August 1888']...
[INIT] NER base embeddings frozen (requires_grad=False)
[INIT] REL base embeddings frozen (requires_grad=False)
[INIT] EL base embeddings frozen (requires_grad=False)
[INIT] NER adapter trainable (requires_grad=True)
[INIT] REL adapter trainable (requires_grad=True)
[INIT] EL adapter trainable (requires_g

In [ ]:
# Load the trained model
from train import load_trained_model, extract_entities

model = load_trained_model("./checkpoints/model_final.pt")

# Extract entities from text
text = "Apple Inc. was founded by Steve Jobs in Cupertino."
entity_types = ["organization", "person", "location"]
results = extract_entities(text, model, tokenizer, entity_types)

for r in results:
    print(f"{r['text']} -> {r['type']} (score: {r['score']:.2f})")

In [ ]:
from datasets import load_dataset
import json

item = next(iter(load_dataset("Universal-NER/Pile-NER-type", split="train", streaming=True)))
print(json.dumps(item, indent=2)[:2000])

{
  "conversations": [
    {
      "from": "human",
      "value": "Text: Q:\n\nPosition character based on enemy coordinates in lua\n\nI have written a function here which should turn my character based on enemy coordinates but it's not perfect because it does not always turn where I want it to and perhaps there is a better way of writing it\nlocal myPosition = {x = 350, y = 355}\nlocal enemyPosition = {x = 352, y = 354}\nlocal xValue, yValue, xDir, yDir, dir\n\nif myPosition.x > enemyPosition.x then\n    xValue = myPosition.x - enemyPosition.x\nelseif myPosition.x < enemyPosition.x then\n    xValue = myPosition.x - enemyPosition.x\nelse\n    xValue = 0\nend\n\nif myPosition.y > enemyPosition.y then\n    yValue = myPosition.y - enemyPosition.y\nelseif myPosition.y < enemyPosition.y then\n    yValue = myPosition.y - enemyPosition.y\nelse\n    yValue = 0\nend\n\nif xValue < 0 then\n    xDir = \"TURN RIGHT\"\nelseif xValue > 0 then\n    xDir = \"TURN LEFT\"\nend\n\nif yValue < 0 then\n  

In [ ]:
!ls -al

total 4705136
drwxr-xr-x 1 root root       4096 Mar 24 15:37 .
drwxr-xr-x 1 root root       4096 Mar 24 12:43 ..
drwxr-xr-x 4 root root       4096 Feb  6 14:31 .config
-rw-r--r-- 1 root root 2409015820 Mar 24 15:06 final_kg_model.pt
-rw-r--r-- 1 root root 2409013822 Mar 24 15:49 kg_model.pt
drwxr-xr-x 1 root root       4096 Feb  6 14:31 sample_data
